# Books / literature domain — `books`

High-quality Wikidata generator for Russian/English multihop benchmark queries in the books/literature domain.

This patched version targets 130 examples across L1–L5: 10 L1, 20 L2, 30 L3, 35 L4, and 35 L5. The dataset still emphasizes L3–L5 (100/130 examples), keeps the public JSONL schema identical to other domains, writes progress incrementally, and can resume from an existing `books.jsonl` file.


## 1. Load common helpers

The notebook can run either after `00_common_helpers.ipynb` or standalone next to `common_helpers.py`.


In [1]:
from pathlib import Path

if "BenchmarkExample" not in globals():
    helper_path = Path("common_helpers.py")
    if not helper_path.exists():
        helper_path = Path("/mnt/data/common_helpers.py")
    exec(helper_path.read_text(encoding="utf-8"), globals())


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Target plan creates 130 records for L1-L5. The runner writes `out_wikidata_benchmark/domain_outputs/books.jsonl` incrementally and resumes safely when `overwrite=False`.


In [2]:
import json
import random
import re
import time
import requests
from collections import Counter, defaultdict
from dataclasses import asdict
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover
    tqdm = None

BOOKS_GENERATOR_VERSION = "v36_130_l1_l5_resumable"
BOOKS_SEED = 20260521
BOOKS_RNG = random.Random(BOOKS_SEED)

BOOKS_TARGET_PLAN: Dict[str, int] = {
    "L1": 10,
    "L2": 20,
    "L3": 30,
    "L4": 35,
    "L5": 35,
}
BOOKS_TARGET_TOTAL = sum(BOOKS_TARGET_PLAN.values())

# Final-quality defaults: fresh WDQS queries, complete accepted gold sets up to 100.
BOOKS_USE_WDQS_CACHE = globals().get("BOOKS_USE_WDQS_CACHE", False)
BOOKS_PROBE_LIMIT = 101
BOOKS_MAX_GOLD_ALLOWED = 100
BOOKS_GOLD_LIMIT = 100
BOOKS_MIN_GOLD = 3
# Public gold is complete under the RU+EN label policy. For quality, skip
# queries where almost all raw Wikidata matches are removed only because
# they lack public RU/EN labels; such examples feel under-complete to users.
BOOKS_MIN_LABEL_RETENTION_RATIO = 0.30
BOOKS_REQUIRE_UNIQUE_PUBLIC_GOLD_LABELS = True
BOOKS_GENERATOR_MAX_ATTEMPTS = 120
BOOKS_PER_TEMPLATE_CANDIDATE_LIMIT = 1200
# Fail-fast WDQS settings for candidate probing. Books queries can occasionally
# hang on broad genre/year combinations; slow candidates are skipped instead of
# blocking one accepted record for 20-30 minutes.
BOOKS_WDQS_FAST_TIMEOUT_SECONDS = 8
BOOKS_WDQS_FAST_MAX_RETRIES = 1
BOOKS_MAX_PREFERRED_GENRE_TRIES_PER_SLOT = 1

# Direct HTTP WDQS profile. This is stronger than mutating wd.timeout: the
# requests layer itself times out, so a single broad genre query cannot block
# the notebook for 20-30 minutes.
BOOKS_WDQS_ENDPOINT = globals().get("BOOKS_WDQS_ENDPOINT", "https://query.wikidata.org/sparql")
BOOKS_WDQS_CONNECT_TIMEOUT_SECONDS = 3.0
BOOKS_WDQS_READ_TIMEOUT_SECONDS = float(BOOKS_WDQS_FAST_TIMEOUT_SECONDS)
BOOKS_WDQS_USER_AGENT = globals().get("BOOKS_WDQS_USER_AGENT", "multihop-benchmark-books/04_books-v28-l3-l5-gender-balanced")
BOOKS_MAX_SECONDS_PER_ACCEPTED_RECORD = 240.0

# Global rejected-cache is useful for L1/L2, but for L3-L5 it can exhaust the
# finite candidate frontier and make the runner fail with mostly
# known_bad_constraints. Keep rejection cache local to one public example.
BOOKS_USE_GLOBAL_REJECTED_SIGNATURE_CACHE = False
BOOKS_DIVERSITY_PROFILE_OFFSETS = 12

# Gender balance for L3-L5. This is a generation-time diversity constraint,
# not a dataset constraint: gendered templates are balanced male/female, while
# non-gender templates remain available for variety.
BOOKS_BALANCE_GENDERED_AUTHOR_QUERIES = True
BOOKS_GENDER_BALANCE_MAX_DIFF = 1
BOOKS_MAX_CONSECUTIVE_SAME_GENDER = 2

# v34: start each level with a small set of WDQS-validated fast seeds.
# These are ordinary candidate specs, not hardcoded gold answers: the notebook
# still reruns WDQS and quality gates before accepting them.
BOOKS_USE_VALIDATED_FAST_SEEDS = True
BOOKS_MAX_FAILED_GENERATE_CALLS_PER_LEVEL = 20
BOOKS_FAILED_SLOT_PROFILE_STRIDE = 17

BOOKS_OUTPUT_DIR = Path(globals().get("BOOKS_OUTPUT_DIR", Path(OUT_DIR) / "domain_outputs"))
BOOKS_OUTPUT_PATH = BOOKS_OUTPUT_DIR / "books.jsonl"
BOOKS_AUDIT_PATH = BOOKS_OUTPUT_DIR / "books.audit.json"
BOOKS_SKIPPED_PATH = BOOKS_OUTPUT_DIR / "books.skipped.jsonl"
BOOKS_CHECKPOINT_PATH = BOOKS_OUTPUT_DIR / "books.checkpoint.json"

RUN_BOOKS_GENERATION = globals().get("RUN_BOOKS_GENERATION", True)


## 3. Low-level helpers

These helpers implement strict label collection, clean constraints, gold completeness gates, and SPARQL validation templates.


In [3]:
_QID_RE_BOOKS = re.compile(r"^Q\d+$")
_CYRILLIC_RE_BOOKS = re.compile(r"[\u0400-\u052F]")


def _bk_escape(s: Any) -> str:
    return str(s).replace("\\", "\\\\").replace('"', '\\"')


def _bk_norm_key(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip()).casefold()


def _bk_fmt_int(n: int) -> str:
    return f"{int(n):,}".replace(",", " ")


def _bk_year_phrase_ru(min_year: Optional[int] = None, max_year: Optional[int] = None) -> str:
    if min_year is not None and max_year is not None:
        return f"с {int(min_year)} по {int(max_year)} год"
    if min_year is not None:
        return f"не ранее {int(min_year)} года"
    if max_year is not None:
        return f"не позднее {int(max_year)} года"
    return ""


def _bk_year_phrase_en(min_year: Optional[int] = None, max_year: Optional[int] = None) -> str:
    if min_year is not None and max_year is not None:
        return f"between {int(min_year)} and {int(max_year)}"
    if min_year is not None:
        return f"no earlier than {int(min_year)}"
    if max_year is not None:
        return f"no later than {int(max_year)}"
    return ""


def _bk_public_constraints_ok(constraints: Dict[str, Any]) -> bool:
    banned = {
        "qid", "template_id", "template_family", "constraint_language",
        "requires_ru_label", "requires_en_label", "requires_latin_script_en_label",
        "max_gold_allowed", "wdqs_candidate_limit", "gold_limit",
    }
    def walk(x: Any) -> bool:
        if isinstance(x, dict):
            for k, v in x.items():
                lk = str(k).lower()
                if lk in banned or lk.endswith("_qid") or lk == "label_en":
                    return False
                if not walk(v):
                    return False
        elif isinstance(x, list):
            return all(walk(v) for v in x)
        return True
    return walk(constraints)


def _bk_clean_constraints(d: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in d.items():
        if v is None:
            continue
        if isinstance(v, bool):
            if v:
                out[k] = v
            continue
        out[k] = v
    assert _bk_public_constraints_ok(out), f"Technical field leaked into public constraints: {out}"
    return out


def _bk_duplicate_labels(labels: Sequence[str]) -> Dict[str, int]:
    c = Counter(_bk_norm_key(x) for x in labels if str(x or "").strip())
    return {k: v for k, v in c.items() if v > 1}


def _bk_answer_label_lines(item_var: str = "item") -> List[str]:
    return [
        f'?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") .',
        f'?{item_var} rdfs:label ?{item_var}LabelEn FILTER(LANG(?{item_var}LabelEn) = "en") .',
        f'FILTER(!REGEX(STR(?{item_var}LabelEn), "[А-Яа-яЁё]")) .',
    ]


def _bk_build_select_query(
    answer_lines: Sequence[str],
    where_lines: Sequence[str],
    item_var: str = "item",
    limit: int = BOOKS_PROBE_LIMIT,
) -> str:
    body = "\n      ".join([str(x).strip() for x in list(answer_lines) + list(where_lines) + _bk_answer_label_lines(item_var) if str(x).strip()])
    return f"""
    SELECT DISTINCT ?{item_var} ?{item_var}LabelRu ?{item_var}LabelEn WHERE {{
      {body}
    }}
    ORDER BY LCASE(STR(?{item_var}LabelEn))
    LIMIT {int(limit)}
    """.strip()


def _bk_build_count_query(
    answer_lines: Sequence[str],
    where_lines: Sequence[str],
    item_var: str = "item",
    require_labels: bool = True,
) -> str:
    # Exact completeness check for the same answer universe used for public gold.
    # require_labels=True means: count only items that can actually appear in
    # gold_answer_labels_ru/en under the benchmark label-quality policy.
    label_lines = _bk_answer_label_lines(item_var) if require_labels else []
    body = "\n      ".join([
        str(x).strip()
        for x in list(answer_lines) + list(where_lines) + list(label_lines)
        if str(x).strip()
    ])
    return f"""
    SELECT (COUNT(DISTINCT ?{item_var}) AS ?count) WHERE {{
      {body}
    }}
    """.strip()


def _bk_exact_count_from_wdqs(query: str) -> int:
    rows = _bk_rows_from_wdqs(query)
    if not rows:
        return 0
    raw = rows[0].get("count", 0)
    try:
        return int(str(raw))
    except Exception:
        return int(float(str(raw)))


def _bk_build_ask_query(answer_lines: Sequence[str], where_lines: Sequence[str], item_var: str = "item") -> str:
    lines = [f"BIND(wd:{{ITEM}} AS ?{item_var})"]
    lines.extend([str(x).strip() for x in list(answer_lines) + list(where_lines) if str(x).strip()])
    body = "\n      ".join(lines)
    return f"""
    ASK WHERE {{
      {body}
    }}
    """.strip()


_BOOKS_QUERY_MEMO: Dict[str, List[Dict[str, str]]] = {}

try:
    WDQSTransientError
except NameError:  # pragma: no cover
    class WDQSTransientError(RuntimeError):
        pass


def _bk_rows_from_select_json(data: Dict[str, Any]) -> List[Dict[str, str]]:
    """Convert raw WDQS JSON bindings into the row format used in this notebook."""
    out: List[Dict[str, str]] = []
    for binding in (data.get("results") or {}).get("bindings", []):
        row: Dict[str, str] = {}
        for key, val in binding.items():
            if isinstance(val, dict):
                row[key] = str(val.get("value", ""))
            else:
                row[key] = str(val or "")
        out.append(row)
    return out


def _bk_rows_from_wdqs(query: str) -> List[Dict[str, str]]:
    """Run WDQS through direct HTTP with a hard per-request timeout.

    Earlier versions tried to set attributes on the project helper ``wd``.
    In some environments that did not interrupt the underlying network call,
    so one broad SPARQL candidate could freeze generation for 20+ minutes.
    This version bypasses the helper for books-domain probing and uses
    ``requests.get(..., timeout=(connect, read))`` directly.
    """
    q = query.strip()
    if q in _BOOKS_QUERY_MEMO:
        return list(_BOOKS_QUERY_MEMO[q])

    params = {
        "query": q,
        "format": "json",
        # WDQS accepts a timeout hint in milliseconds; HTTP timeout below is
        # still the hard guard if the endpoint ignores this hint.
        "timeout": str(int(max(1.0, float(BOOKS_WDQS_READ_TIMEOUT_SECONDS)) * 1000)),
    }
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": str(BOOKS_WDQS_USER_AGENT),
    }

    last_error: Optional[BaseException] = None
    attempts = max(1, int(BOOKS_WDQS_FAST_MAX_RETRIES))
    for attempt in range(attempts):
        try:
            resp = requests.get(
                str(BOOKS_WDQS_ENDPOINT),
                params=params,
                headers=headers,
                timeout=(float(BOOKS_WDQS_CONNECT_TIMEOUT_SECONDS), float(BOOKS_WDQS_READ_TIMEOUT_SECONDS)),
            )
            if resp.status_code in {429, 500, 502, 503, 504}:
                raise WDQSTransientError(f"WDQS HTTP {resp.status_code}")
            resp.raise_for_status()
            rows = _bk_rows_from_select_json(resp.json())
            _BOOKS_QUERY_MEMO[q] = list(rows)
            return rows
        except Exception as e:
            last_error = e
            # One retry max by default; tiny backoff only for transient errors.
            if attempt + 1 < attempts:
                time.sleep(0.35)

    raise WDQSTransientError(f"WDQS direct fail-fast skip: {type(last_error).__name__}: {last_error}")


def _bk_collect_gold(
    answer_lines: Sequence[str],
    where_lines: Sequence[str],
    requested_count: int,
    template_id: str,
    template_family: str,
    item_var: str = "item",
) -> Tuple[str, str, Optional[Dict[str, Any]], List[Tuple[str, str, str]]]:
    """Collect a complete public gold set quickly.

    Fast v13 strategy:
    1) Run the public-gold SELECT with LIMIT 101 first.
       If fewer than 101 rows are returned, WDQS exhausted the public universe,
       so len(distinct rows) is the exact public count under the RU+EN label policy.
    2) Only for candidates that pass public-count and duplicate-label gates, run
       the raw no-label COUNT to compute label-retention metadata.
    This avoids the old 2 COUNT + 1 SELECT pattern for every failed candidate.
    """
    sparql = _bk_build_select_query(answer_lines, where_lines, item_var=item_var, limit=BOOKS_PROBE_LIMIT)
    ask = _bk_build_ask_query(answer_lines, where_lines, item_var=item_var)

    rows = _bk_rows_from_wdqs(sparql)

    if len(rows) >= BOOKS_PROBE_LIMIT:
        meta = {
            "source": "wikidata_sparql",
            "constraints_are_wdqs_only": True,
            "wdqs_candidate_limit": BOOKS_PROBE_LIMIT,
            "max_gold_allowed": BOOKS_MAX_GOLD_ALLOWED,
            "skip_if_rows_returned_reach_limit": True,
            "skip_if_gold_count_gt": BOOKS_MAX_GOLD_ALLOWED,
            "public_gold_policy": "complete among Wikidata items matching constraints and having both ru/en labels",
            "rows_returned_by_wdqs_after_label_filter": len(rows),
            "gold_may_be_incomplete_due_to_wdqs_limit": True,
            "skip_reason": "select_hit_probe_limit",
            "template_id": template_id,
            "template_family": template_family,
        }
        return sparql, ask, meta, []

    triples: List[Tuple[str, str, str]] = []
    seen = set()
    for r in rows:
        qid = uri_to_qid(r.get(item_var, ""))
        ru = str(r.get(f"{item_var}LabelRu") or "").strip()
        en = str(r.get(f"{item_var}LabelEn") or "").strip()
        if not qid or not ru or not en:
            continue
        if _CYRILLIC_RE_BOOKS.search(en):
            continue
        if qid in seen:
            continue
        triples.append((qid, ru, en))
        seen.add(qid)

    exact_count_after_label_filter = len(triples)

    if exact_count_after_label_filter > BOOKS_MAX_GOLD_ALLOWED:
        meta = {
            "source": "wikidata_sparql",
            "constraints_are_wdqs_only": True,
            "wdqs_candidate_limit": BOOKS_PROBE_LIMIT,
            "max_gold_allowed": BOOKS_MAX_GOLD_ALLOWED,
            "wdqs_exact_count_after_label_filter": exact_count_after_label_filter,
            "public_gold_count_source": "exhaustive_select_under_probe_limit",
            "public_gold_policy": "complete among Wikidata items matching constraints and having both ru/en labels",
            "skip_if_gold_count_gt": BOOKS_MAX_GOLD_ALLOWED,
            "skip_reason": "too_many_gold_select_count",
            "gold_may_be_incomplete_due_to_wdqs_limit": False,
            "template_id": template_id,
            "template_family": template_family,
        }
        return sparql, ask, meta, []

    if exact_count_after_label_filter < int(requested_count):
        meta = {
            "source": "wikidata_sparql",
            "constraints_are_wdqs_only": True,
            "wdqs_candidate_limit": BOOKS_PROBE_LIMIT,
            "max_gold_allowed": BOOKS_MAX_GOLD_ALLOWED,
            "wdqs_exact_count_after_label_filter": exact_count_after_label_filter,
            "public_gold_count_source": "exhaustive_select_under_probe_limit",
            "public_gold_policy": "complete among Wikidata items matching constraints and having both ru/en labels",
            "rows_returned_by_wdqs_after_label_filter": len(rows),
            "gold_returned": exact_count_after_label_filter,
            "skip_reason": "not_enough_gold_select_count",
            "template_id": template_id,
            "template_family": template_family,
        }
        return sparql, ask, meta, []

    ru_dups = _bk_duplicate_labels([ru for _, ru, _ in triples])
    en_dups = _bk_duplicate_labels([en for _, _, en in triples])

    if BOOKS_REQUIRE_UNIQUE_PUBLIC_GOLD_LABELS and (ru_dups or en_dups):
        meta = {
            "source": "wikidata_sparql",
            "constraints_are_wdqs_only": True,
            "wdqs_candidate_limit": BOOKS_PROBE_LIMIT,
            "max_gold_allowed": BOOKS_MAX_GOLD_ALLOWED,
            "skip_if_rows_returned_reach_limit": True,
            "skip_if_gold_count_gt": BOOKS_MAX_GOLD_ALLOWED,
            "requires_both_ru_and_en_labels": True,
            "requires_latin_script_en_label": True,
            "requires_unique_gold_labels": True,
            "duplicate_public_gold_labels": {"ru": ru_dups, "en": en_dups},
            "wdqs_exact_count_after_label_filter": exact_count_after_label_filter,
            "public_gold_count_source": "exhaustive_select_under_probe_limit",
            "public_gold_policy": "complete among Wikidata items matching constraints and having both ru/en labels",
            "rows_returned_by_wdqs_after_label_filter": len(rows),
            "gold_returned": exact_count_after_label_filter,
            "skip_reason": "duplicate_public_gold_labels",
            "template_id": template_id,
            "template_family": template_family,
        }
        return sparql, ask, meta, []

    # Only now pay for the raw count needed to estimate how much of Wikidata was
    # removed solely because public RU+EN labels were missing.
    raw_count_sparql = _bk_build_count_query(answer_lines, where_lines, item_var=item_var, require_labels=False)
    exact_count_without_label_filter = _bk_exact_count_from_wdqs(raw_count_sparql)
    label_filter_removed_count = max(0, exact_count_without_label_filter - exact_count_after_label_filter)
    label_retention_ratio = (
        exact_count_after_label_filter / exact_count_without_label_filter
        if exact_count_without_label_filter > 0
        else 1.0
    )

    if (
        exact_count_after_label_filter >= int(requested_count)
        and exact_count_without_label_filter > 0
        and label_retention_ratio < BOOKS_MIN_LABEL_RETENTION_RATIO
    ):
        meta = {
            "source": "wikidata_sparql",
            "constraints_are_wdqs_only": True,
            "wdqs_candidate_limit": BOOKS_PROBE_LIMIT,
            "max_gold_allowed": BOOKS_MAX_GOLD_ALLOWED,
            "wdqs_exact_count_after_label_filter": exact_count_after_label_filter,
            "wdqs_exact_count_without_label_filter": exact_count_without_label_filter,
            "label_filter_removed_count": label_filter_removed_count,
            "label_retention_ratio": label_retention_ratio,
            "min_label_retention_ratio": BOOKS_MIN_LABEL_RETENTION_RATIO,
            "public_gold_count_source": "exhaustive_select_under_probe_limit",
            "public_gold_policy": "complete among Wikidata items matching constraints and having both ru/en labels",
            "rows_returned_by_wdqs_after_label_filter": len(rows),
            "skip_reason": "low_label_retention",
            "gold_may_be_incomplete_due_to_wdqs_limit": False,
            "template_id": template_id,
            "template_family": template_family,
        }
        return sparql, ask, meta, []

    meta = {
        "source": "wikidata_sparql",
        "constraints_are_wdqs_only": True,
        "wdqs_candidate_limit": BOOKS_PROBE_LIMIT,
        "max_gold_allowed": BOOKS_MAX_GOLD_ALLOWED,
        "skip_if_rows_returned_reach_limit": True,
        "skip_if_gold_count_gt": BOOKS_MAX_GOLD_ALLOWED,
        "requires_both_ru_and_en_labels": True,
        "requires_latin_script_en_label": True,
        "requires_unique_gold_labels": bool(BOOKS_REQUIRE_UNIQUE_PUBLIC_GOLD_LABELS),
        "duplicate_public_gold_labels": {"ru": ru_dups, "en": en_dups},
        "wdqs_exact_count_after_label_filter": exact_count_after_label_filter,
        "wdqs_exact_count_without_label_filter": exact_count_without_label_filter,
        "label_filter_removed_count": label_filter_removed_count,
        "label_retention_ratio": label_retention_ratio,
        "min_label_retention_ratio": BOOKS_MIN_LABEL_RETENTION_RATIO,
        "public_gold_count_source": "exhaustive_select_under_probe_limit",
        "public_gold_policy": "complete among Wikidata items matching constraints and having both ru/en labels",
        "rows_returned_by_wdqs_after_label_filter": len(rows),
        "gold_limit": BOOKS_GOLD_LIMIT,
        "gold_returned": len(triples),
        "gold_total_before_limit": exact_count_after_label_filter,
        "gold_may_be_incomplete_due_to_wdqs_limit": False,
        "gold_truncated_by_local_limit": False,
        "label_sources": {"ru_label": len(triples), "en_label": len(triples)},
        "template_id": template_id,
        "template_family": template_family,
    }
    return sparql, ask, meta, triples


## 4. Domain vocabulary

QIDs are explicit for reproducibility. Public constraints use only English labels.


In [4]:
# Core classes
Q_LITERARY_WORK = "Q7725634"
Q_BOOK = "Q571"
Q_NOVEL = "Q8261"
Q_SHORT_STORY = "Q49084"
Q_NOVELLA = "Q149537"
Q_EDITION_OR_TRANSLATION = "Q3331189"
Q_BOOK_SERIES = "Q277759"
Q_SERIES_OF_CREATIVE_WORKS = "Q7725310"
Q_WEBSITE = "Q35127"
Q_WIKI = "Q171"
Q_HUMAN = "Q5"
Q_WRITER = "Q36180"
Q_FILM = "Q11424"
Q_TV_SERIES = "Q5398426"

# Properties frequently used below:
# P31 instance of, P279 subclass of, P50 author, P136 genre, P407 language of work/name,
# P577 publication date, P27 country of citizenship, P21 sex/gender, P106 occupation,
# P166 award received, P144 based on, P840 narrative location, P179 part of series.

Q_FEMALE = "Q6581072"
Q_MALE = "Q6581097"

GENRES: Dict[str, Dict[str, str]] = {
    "science_fiction": {"qid": "Q24925", "en": "science fiction", "ru": "научная фантастика"},
    "fantasy": {"qid": "Q132311", "en": "fantasy", "ru": "фэнтези"},
    "detective_fiction": {"qid": "Q186424", "en": "detective fiction", "ru": "детективная литература"},
    "horror_fiction": {"qid": "Q200092", "en": "horror fiction", "ru": "литература ужасов"},
    "historical_novel": {"qid": "Q192239", "en": "historical novel", "ru": "исторический роман"},
    "dystopian_fiction": {"qid": "Q1195599", "en": "dystopian fiction", "ru": "антиутопия"},
    "children_literature": {"qid": "Q131539", "en": "children's literature", "ru": "детская литература"},
    "crime_fiction": {"qid": "Q959790", "en": "crime fiction", "ru": "криминальная литература"},
    "adventure_fiction": {"qid": "Q2143665", "en": "adventure fiction", "ru": "приключенческая литература"},
    "romance_novel": {"qid": "Q1054574", "en": "romance novel", "ru": "любовный роман"},
}

COUNTRIES: Dict[str, Dict[str, str]] = {
    "russia": {"qid": "Q159", "en": "Russia", "ru": "Россия"},
    "united_states": {"qid": "Q30", "en": "United States", "ru": "США"},
    "united_kingdom": {"qid": "Q145", "en": "United Kingdom", "ru": "Великобритания"},
    "france": {"qid": "Q142", "en": "France", "ru": "Франция"},
    "germany": {"qid": "Q183", "en": "Germany", "ru": "Германия"},
    "japan": {"qid": "Q17", "en": "Japan", "ru": "Япония"},
    "canada": {"qid": "Q16", "en": "Canada", "ru": "Канада"},
    "italy": {"qid": "Q38", "en": "Italy", "ru": "Италия"},
    "spain": {"qid": "Q29", "en": "Spain", "ru": "Испания"},
    "ireland": {"qid": "Q27", "en": "Ireland", "ru": "Ирландия"},
    "poland": {"qid": "Q36", "en": "Poland", "ru": "Польша"},
    "argentina": {"qid": "Q414", "en": "Argentina", "ru": "Аргентина"},
}

LANGUAGES: Dict[str, Dict[str, str]] = {
    "english": {"qid": "Q1860", "en": "English", "ru": "английский язык"},
    "russian": {"qid": "Q7737", "en": "Russian", "ru": "русский язык"},
    "french": {"qid": "Q150", "en": "French", "ru": "французский язык"},
    "german": {"qid": "Q188", "en": "German", "ru": "немецкий язык"},
    "japanese": {"qid": "Q5287", "en": "Japanese", "ru": "японский язык"},
    "spanish": {"qid": "Q1321", "en": "Spanish", "ru": "испанский язык"},
    "italian": {"qid": "Q652", "en": "Italian", "ru": "итальянский язык"},
    "polish": {"qid": "Q809", "en": "Polish", "ru": "польский язык"},
}

AWARDS: Dict[str, Dict[str, str]] = {
    "nobel_literature": {"qid": "Q37922", "en": "Nobel Prize in Literature", "ru": "Нобелевская премия по литературе"},
    "hugo_best_novel": {"qid": "Q255032", "en": "Hugo Award for Best Novel", "ru": "премия «Хьюго» за лучший роман"},
    "pulitzer_fiction": {"qid": "Q104438", "en": "Pulitzer Prize for Fiction", "ru": "Пулитцеровская премия за художественную книгу"},
    "booker_prize": {"qid": "Q160082", "en": "Booker Prize", "ru": "Букеровская премия"},
}

CITIES: Dict[str, Dict[str, str]] = {
    "london": {"qid": "Q84", "en": "London", "ru": "Лондон"},
    "paris": {"qid": "Q90", "en": "Paris", "ru": "Париж"},
    "moscow": {"qid": "Q649", "en": "Moscow", "ru": "Москва"},
    "saint_petersburg": {"qid": "Q656", "en": "Saint Petersburg", "ru": "Санкт-Петербург"},
    "new_york_city": {"qid": "Q60", "en": "New York City", "ru": "Нью-Йорк"},
    "dublin": {"qid": "Q1761", "en": "Dublin", "ru": "Дублин"},
    "tokyo": {"qid": "Q1490", "en": "Tokyo", "ru": "Токио"},
    "berlin": {"qid": "Q64", "en": "Berlin", "ru": "Берлин"},
}

GENDERS: Dict[str, Dict[str, str]] = {
    "female": {"qid": Q_FEMALE, "en": "female", "ru": "женщина"},
    "male": {"qid": Q_MALE, "en": "male", "ru": "мужчина"},
}


def _bk_ent(table: Dict[str, Dict[str, str]], key: str) -> Dict[str, str]:
    v = table[key]
    assert _QID_RE_BOOKS.fullmatch(v["qid"]), f"Bad QID for {key}: {v}"
    return v


def _bk_literary_work_answer_lines(var: str = "item") -> List[str]:
    # Standalone literary works only. Wikidata sometimes marks franchises,
    # wikis, collaborative universes or series as literary works; those are
    # too noisy for public gold lists in this benchmark.
    return [
        f"?{var} wdt:P31/wdt:P279* wd:{Q_LITERARY_WORK} .",
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_EDITION_OR_TRANSLATION} . }}",
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_BOOK_SERIES} . }}",
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_SERIES_OF_CREATIVE_WORKS} . }}",
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_WEBSITE} . }}",
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_WIKI} . }}",
    ]


def _bk_writer_answer_lines(var: str = "item") -> List[str]:
    return [
        f"?{var} wdt:P31 wd:{Q_HUMAN} .",
        f"?{var} wdt:P106/wdt:P279* wd:{Q_WRITER} .",
    ]


## 5. Template task objects and NLG

Templates produce a task object; the shared collector turns it into a `BenchmarkExample` only if the WDQS gold set is complete and high quality.


In [5]:
class BookTask:
    def __init__(
        self,
        *,
        template_id: str,
        template_family: str,
        answer_kind: str,
        requested_count: int,
        query_text_ru: str,
        query_text_en: str,
        constraints: Dict[str, Any],
        answer_lines: Sequence[str],
        where_lines: Sequence[str],
        is_advanced: bool = False,
    ):
        self.template_id = template_id
        self.template_family = template_family
        self.answer_kind = answer_kind
        self.requested_count = requested_count
        self.query_text_ru = query_text_ru
        self.query_text_en = query_text_en
        self.constraints = _bk_clean_constraints(constraints)
        self.answer_lines = list(answer_lines)
        self.where_lines = list(where_lines)
        self.is_advanced = bool(is_advanced)



def _bk_missing_required_sparql_tokens(task: BookTask, sparql: str) -> List[str]:
    """Guard against stale definitions / old-kernel generation.

    Accepted records must include the strict filters that fix known books-domain
    leakage bugs: non-standalone work noise, earliest publication year, and
    strict novel requirements for novel genres such as historical novel.
    """
    s = str(sparql or "")
    missing: List[str] = []

    answer_s = "\n".join(task.answer_lines)
    if f"wd:{Q_LITERARY_WORK}" in answer_s:
        required_work_noise_filters = [
            Q_EDITION_OR_TRANSLATION,
            Q_BOOK_SERIES,
            Q_SERIES_OF_CREATIVE_WORKS,
            Q_WEBSITE,
            Q_WIKI,
        ]
        for qid in required_work_noise_filters:
            if f"wd:{qid}" not in s:
                missing.append(f"missing_work_noise_filter_{qid}")

    has_year_filter = (
        "publication_year_min" in (task.constraints or {})
        or "publication_year_max" in (task.constraints or {})
    )
    if has_year_filter and ("FILTER NOT EXISTS" not in s or "?earlierPublicationDate" not in s):
        missing.append("missing_earliest_publication_year_filter")

    genre_name = str((task.constraints or {}).get("genre", ""))
    if genre_name.endswith("novel"):
        if (task.constraints or {}).get("kind") != "novel":
            missing.append("novel_genre_public_kind_not_novel")
        for qid in (Q_NOVEL, Q_SHORT_STORY, Q_NOVELLA):
            if f"wd:{qid}" not in s:
                missing.append(f"missing_strict_novel_filter_{qid}")

    return missing



def _bk_make_example(task: BookTask, complexity: str, idx: int, attempt: int) -> Optional[BenchmarkExample]:
    try:
        sparql, ask, meta, gold = _bk_collect_gold(
            task.answer_lines,
            task.where_lines,
            requested_count=task.requested_count,
            template_id=task.template_id,
            template_family=task.template_family,
            item_var="item",
        )
    except WDQSTransientError:
        return None
    except Exception as e:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[books] template {task.template_id} failed: {e}")
        return None

    missing_filter_tokens = _bk_missing_required_sparql_tokens(task, sparql)
    if missing_filter_tokens:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[books] stale/unsafe SPARQL for {task.template_id}: {missing_filter_tokens}")
        return None

    if not gold:
        return None

    qids = [q for q, _, _ in gold]
    labels_ru = [ru for _, ru, _ in gold]
    labels_en = [en for _, _, en in gold]
    if not (len(qids) == len(labels_ru) == len(labels_en)):
        return None
    if len(qids) > BOOKS_MAX_GOLD_ALLOWED:
        return None
    if any(_CYRILLIC_RE_BOOKS.search(x or "") for x in labels_en):
        return None

    meta = dict(meta or {})
    if "wdqs_exact_count_after_label_filter" not in meta:
        return None
    if meta.get("gold_returned") != len(qids):
        return None
    if meta.get("gold_total_before_limit") != len(qids):
        return None
    meta.update({
        "generator_version": BOOKS_GENERATOR_VERSION,
        "strict_sparql_filter_guard": True,
        "generator_attempts_for_record": int(attempt),
        "answer_kind": task.answer_kind,
    })

    return BenchmarkExample(
        id=f"books_{complexity.lower()}_{idx:04d}",
        domain="books",
        complexity=complexity,
        query_text_ru=task.query_text_ru,
        query_text_en=task.query_text_en,
        constraints=task.constraints,
        requested_count=task.requested_count,
        gold_answer_qids=qids,
        gold_answer_labels_ru=labels_ru,
        gold_answer_labels_en=labels_en,
        sparql_query=sparql,
        created_at=utc_now_z(),
        is_advanced=task.is_advanced,
        template_id=task.template_id,
        template_family=task.template_family,
        gold_truncated=False,
        ask_validator_sparql=ask,
        gold_collection_meta=meta,
    )


def _bk_date_lines(var: str = "item", min_year: Optional[int] = None, max_year: Optional[int] = None) -> List[str]:
    if min_year is None and max_year is None:
        return []
    # Use the earliest available publication date, not any later edition/reprint.
    # Without this guard, a work first published in 1968/1969 can slip into a
    # 1970+ query because Wikidata also stores a later book/translation date.
    out = [
        f"?{var} wdt:P577 ?publicationDate .",
        "BIND(YEAR(?publicationDate) AS ?publicationYear) .",
        f"FILTER NOT EXISTS {{ ?{var} wdt:P577 ?earlierPublicationDate . FILTER(YEAR(?earlierPublicationDate) < ?publicationYear) }}",
    ]
    if min_year is not None:
        out.append(f"FILTER(?publicationYear >= {int(min_year)}) .")
    if max_year is not None:
        out.append(f"FILTER(?publicationYear <= {int(max_year)}) .")
    return out


def _bk_work_common_lines(
    *,
    genre: Optional[Dict[str, str]] = None,
    language: Optional[Dict[str, str]] = None,
    author_country: Optional[Dict[str, str]] = None,
    author_gender: Optional[Dict[str, str]] = None,
    author_award: Optional[Dict[str, str]] = None,
    min_year: Optional[int] = None,
    max_year: Optional[int] = None,
    not_series: bool = False,
) -> List[str]:
    lines: List[str] = []
    needs_author = any(x is not None for x in (author_country, author_gender, author_award))
    if needs_author:
        lines.append("?item wdt:P50 ?author .")
        lines.append(f"?author wdt:P106/wdt:P279* wd:{Q_WRITER} .")
    if genre is not None:
        lines.append(f"?item wdt:P136/wdt:P279* wd:{genre['qid']} .")
        if str(genre.get("en", "")).endswith("novel"):
            lines.append(f"?item wdt:P31/wdt:P279* wd:{Q_NOVEL} .")
            lines.append(f"MINUS {{ ?item wdt:P31/wdt:P279* wd:{Q_SHORT_STORY} . }}")
            lines.append(f"MINUS {{ ?item wdt:P31/wdt:P279* wd:{Q_NOVELLA} . }}")
    if language is not None:
        lines.append(f"?item wdt:P407 wd:{language['qid']} .")
    if author_country is not None:
        lines.append(f"?author wdt:P27 wd:{author_country['qid']} .")
    if author_gender is not None:
        lines.append(f"?author wdt:P21 wd:{author_gender['qid']} .")
    if author_award is not None:
        lines.append(f"?author wdt:P166 wd:{author_award['qid']} .")
    lines.extend(_bk_date_lines("item", min_year, max_year))
    if not_series:
        lines.append("MINUS { ?item wdt:P179 ?series . }")
    return lines


def _bk_writer_common_lines(
    *,
    citizenship: Optional[Dict[str, str]] = None,
    gender: Optional[Dict[str, str]] = None,
    award: Optional[Dict[str, str]] = None,
    work_genre: Optional[Dict[str, str]] = None,
    work_language: Optional[Dict[str, str]] = None,
    work_min_year: Optional[int] = None,
    work_max_year: Optional[int] = None,
    adapted_work_after_year: Optional[int] = None,
) -> List[str]:
    lines: List[str] = []
    if citizenship is not None:
        lines.append(f"?item wdt:P27 wd:{citizenship['qid']} .")
    if gender is not None:
        lines.append(f"?item wdt:P21 wd:{gender['qid']} .")
    if award is not None:
        lines.append(f"?item wdt:P166 wd:{award['qid']} .")
    if any(x is not None for x in (work_genre, work_language, work_min_year, work_max_year, adapted_work_after_year)):
        lines.append("?work wdt:P50 ?item .")
        lines.append(f"?work wdt:P31/wdt:P279* wd:{Q_LITERARY_WORK} .")
        lines.append(f"MINUS {{ ?work wdt:P31/wdt:P279* wd:{Q_EDITION_OR_TRANSLATION} . }}")
        if work_genre is not None:
            lines.append(f"?work wdt:P136/wdt:P279* wd:{work_genre['qid']} .")
        if work_language is not None:
            lines.append(f"?work wdt:P407 wd:{work_language['qid']} .")
        lines.extend(_bk_date_lines("work", work_min_year, work_max_year))
        if adapted_work_after_year is not None:
            lines.extend([
                "?adaptation wdt:P144 ?work .",
                f"VALUES ?adaptationType {{ wd:{Q_FILM} wd:{Q_TV_SERIES} }}",
                "?adaptation wdt:P31/wdt:P279* ?adaptationType .",
                "?adaptation wdt:P577 ?adaptationDate .",
                f"FILTER(YEAR(?adaptationDate) >= {int(adapted_work_after_year)}) .",
            ])
    return lines


def _bk_constraints_work_base(
    *,
    genre: Optional[Dict[str, str]] = None,
    language: Optional[Dict[str, str]] = None,
    author_country: Optional[Dict[str, str]] = None,
    author_gender: Optional[Dict[str, str]] = None,
    author_award: Optional[Dict[str, str]] = None,
    min_year: Optional[int] = None,
    max_year: Optional[int] = None,
    not_series: bool = False,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    d: Dict[str, Any] = {"kind": "novel" if genre and str(genre.get("en", "")).endswith("novel") else "literary work"}
    if genre: d["genre"] = genre["en"]
    if language: d["language"] = language["en"]
    if author_country: d["author_citizenship"] = author_country["en"]
    if author_gender: d["author_gender"] = author_gender["en"]
    if author_award: d["author_award_received"] = author_award["en"]
    if min_year is not None: d["publication_year_min"] = int(min_year)
    if max_year is not None: d["publication_year_max"] = int(max_year)
    if not_series: d["not_part_of_series"] = True
    if extra: d.update(extra)
    return _bk_clean_constraints(d)


def _bk_constraints_writer_base(
    *,
    citizenship: Optional[Dict[str, str]] = None,
    gender: Optional[Dict[str, str]] = None,
    award: Optional[Dict[str, str]] = None,
    work_genre: Optional[Dict[str, str]] = None,
    work_language: Optional[Dict[str, str]] = None,
    work_min_year: Optional[int] = None,
    work_max_year: Optional[int] = None,
    adapted_work_after_year: Optional[int] = None,
) -> Dict[str, Any]:
    d: Dict[str, Any] = {"kind": "writer"}
    if citizenship: d["citizenship"] = citizenship["en"]
    if gender: d["gender"] = gender["en"]
    if award: d["award_received"] = award["en"]
    if work_genre: d["has_authored_work_genre"] = work_genre["en"]
    if work_language: d["has_authored_work_language"] = work_language["en"]
    if work_min_year is not None: d["has_authored_work_publication_year_min"] = int(work_min_year)
    if work_max_year is not None: d["has_authored_work_publication_year_max"] = int(work_max_year)
    if adapted_work_after_year is not None: d["has_authored_work_with_screen_adaptation_year_min"] = int(adapted_work_after_year)
    return _bk_clean_constraints(d)


## 6. Template builders

Each builder creates one candidate task from clean semantic parameters.


In [6]:

BOOKS_DEFAULT_LANGUAGE_BY_COUNTRY: Dict[str, str] = {
    "united_states": "english",
    "united_kingdom": "english",
    "canada": "english",
    "ireland": "english",
    "russia": "russian",
    "france": "french",
    "germany": "german",
    "japan": "japanese",
    "italy": "italian",
    "spain": "spanish",
    "poland": "polish",
    "argentina": "spanish",
}

def _bk_default_language_for_country(country_key: str) -> Optional[str]:
    return BOOKS_DEFAULT_LANGUAGE_BY_COUNTRY.get(str(country_key))

def _task_work_genre_language_year(
    *, genre_key: str, lang_key: str, min_year: Optional[int], max_year: Optional[int], k: int
) -> BookTask:
    genre = _bk_ent(GENRES, genre_key)
    lang = _bk_ent(LANGUAGES, lang_key)
    has_year = (min_year is not None or max_year is not None)
    if has_year:
        yru = _bk_year_phrase_ru(min_year, max_year)
        yen = _bk_year_phrase_en(min_year, max_year)
        qru = f"Назови {k} литературных произведений жанра «{genre['ru']}» на языке: {lang['ru']}, опубликованных {yru}."
        qen = f"Name {k} literary works in the {genre['en']} genre whose language is {lang['en']} and that were published {yen}."
    else:
        qru = f"Назови {k} литературных произведений жанра «{genre['ru']}» на языке: {lang['ru']}."
        qen = f"Name {k} literary works in the {genre['en']} genre whose language is {lang['en']}."
    return BookTask(
        template_id="books_work_genre_language_year",
        template_family="works_basic",
        answer_kind="literary work",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_work_base(genre=genre, language=lang, min_year=min_year, max_year=max_year),
        answer_lines=_bk_literary_work_answer_lines("item"),
        where_lines=_bk_work_common_lines(genre=genre, language=lang, min_year=min_year, max_year=max_year),
    )

def _task_work_author_country_genre_year(
    *, country_key: str, genre_key: str, min_year: Optional[int], max_year: Optional[int], k: int, lang_key: Optional[str] = None
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    genre = _bk_ent(GENRES, genre_key)
    # For author-citizenship templates we also constrain the work language.
    # This prevents semantically strange gold such as English/Russian works
    # leaking into "authors with citizenship of France" only because of noisy
    # or unexpected Wikidata author/citizenship links.
    lang_key = lang_key or _bk_default_language_for_country(country_key)
    lang = _bk_ent(LANGUAGES, lang_key) if lang_key else None

    parts_ru = [f"жанра «{genre['ru']}»"]
    if lang is not None:
        parts_ru.append(f"на языке: {lang['ru']}")
    has_year = (min_year is not None or max_year is not None)
    if has_year:
        parts_ru.append(f"опубликованных {_bk_year_phrase_ru(min_year, max_year)}")
    parts_ru.append(f"у авторов с гражданством: {country['ru']}")
    qru = f"Назови {k} литературных произведений " + ", ".join(parts_ru) + "."

    parts_en = [f"in the {genre['en']} genre"]
    if lang is not None:
        parts_en.append(f"whose language is {lang['en']}")
    if has_year:
        parts_en.append(f"published {_bk_year_phrase_en(min_year, max_year)}")
    parts_en.append(f"whose authors have citizenship of {country['en']}")
    qen = f"Name {k} literary works " + ", ".join(parts_en) + "."

    return BookTask(
        template_id="books_work_author_country_genre_year",
        template_family="works_author",
        answer_kind="literary work",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_work_base(genre=genre, language=lang, author_country=country, min_year=min_year, max_year=max_year),
        answer_lines=_bk_literary_work_answer_lines("item"),
        where_lines=_bk_work_common_lines(genre=genre, language=lang, author_country=country, min_year=min_year, max_year=max_year),
    )

def _task_work_author_country_language_year(
    *, country_key: str, lang_key: str, min_year: Optional[int], max_year: Optional[int], k: int
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    lang = _bk_ent(LANGUAGES, lang_key)
    has_year = (min_year is not None or max_year is not None)
    if has_year:
        yru = _bk_year_phrase_ru(min_year, max_year)
        yen = _bk_year_phrase_en(min_year, max_year)
        qru = f"Назови {k} литературных произведений на языке: {lang['ru']}, опубликованных {yru}, у авторов с гражданством: {country['ru']}."
        qen = f"Name {k} literary works whose language is {lang['en']}, published {yen}, and whose authors have citizenship of {country['en']}."
    else:
        qru = f"Назови {k} литературных произведений на языке: {lang['ru']} у авторов с гражданством: {country['ru']}."
        qen = f"Name {k} literary works whose language is {lang['en']} and whose authors have citizenship of {country['en']}."
    return BookTask(
        template_id="books_work_author_country_language_year",
        template_family="works_author",
        answer_kind="literary work",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_work_base(language=lang, author_country=country, min_year=min_year, max_year=max_year),
        answer_lines=_bk_literary_work_answer_lines("item"),
        where_lines=_bk_work_common_lines(language=lang, author_country=country, min_year=min_year, max_year=max_year),
    )

def _task_work_author_gender_country_genre_language_year(
    *, country_key: str, gender_key: str, genre_key: str, lang_key: str, min_year: Optional[int], max_year: Optional[int], k: int
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    genre = _bk_ent(GENRES, genre_key)
    lang = _bk_ent(LANGUAGES, lang_key)
    gender = _bk_ent(GENDERS, gender_key)
    yru = _bk_year_phrase_ru(min_year, max_year)
    yen = _bk_year_phrase_en(min_year, max_year)
    author_phrase_ru = "женщин-авторов" if gender_key == "female" else "мужчин-авторов"
    author_phrase_en = "female citizens" if gender_key == "female" else "male citizens"
    qru = f"Назови {k} литературных произведений жанра «{genre['ru']}» на языке: {lang['ru']}, опубликованных {yru}, у {author_phrase_ru} с гражданством: {country['ru']}."
    qen = f"Name {k} literary works in the {genre['en']} genre whose language is {lang['en']}, published {yen}, and whose authors are {author_phrase_en} of {country['en']}."
    return BookTask(
        template_id=f"books_work_author_{gender_key}_country_genre_language_year",
        template_family="works_author_gender",
        answer_kind="literary work",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_work_base(genre=genre, language=lang, author_country=country, author_gender=gender, min_year=min_year, max_year=max_year),
        answer_lines=_bk_literary_work_answer_lines("item"),
        where_lines=_bk_work_common_lines(genre=genre, language=lang, author_country=country, author_gender=gender, min_year=min_year, max_year=max_year),
        is_advanced=True,
    )


def _task_work_female_author_country_genre_language_year(
    *, country_key: str, genre_key: str, lang_key: str, min_year: Optional[int], max_year: Optional[int], k: int
) -> BookTask:
    # Backward-compatible wrapper for older candidate specs. New specs use
    # work_author_gender_country_genre_language_year with explicit gender_key.
    return _task_work_author_gender_country_genre_language_year(
        country_key=country_key, gender_key="female", genre_key=genre_key,
        lang_key=lang_key, min_year=min_year, max_year=max_year, k=k,
    )


def _task_writer_citizenship_gender_work_genre_language(
    *, country_key: str, gender_key: str, genre_key: str, lang_key: str, k: int,
    min_year: Optional[int] = None, max_year: Optional[int] = None,
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    gender = _bk_ent(GENDERS, gender_key)
    genre = _bk_ent(GENRES, genre_key)
    lang = _bk_ent(LANGUAGES, lang_key)
    yru = _bk_year_phrase_ru(min_year, max_year)
    yen = _bk_year_phrase_en(min_year, max_year)
    year_ru = f", опубликованное {yru}" if (min_year is not None or max_year is not None) else ""
    year_en = f", published {yen}" if (min_year is not None or max_year is not None) else ""
    qru = f"Назови {k} писателей с гражданством: {country['ru']} и полом: {gender['ru']}, у которых есть произведение жанра «{genre['ru']}» на языке: {lang['ru']}{year_ru}."
    qen = f"Name {k} writers whose citizenship is {country['en']} and gender is {gender['en']}, and who authored a {genre['en']} work whose language is {lang['en']}{year_en}."
    return BookTask(
        template_id="books_writer_citizenship_gender_work_genre_language",
        template_family="writers_work_based",
        answer_kind="writer",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_writer_base(citizenship=country, gender=gender, work_genre=genre, work_language=lang, work_min_year=min_year, work_max_year=max_year),
        answer_lines=_bk_writer_answer_lines("item"),
        where_lines=_bk_writer_common_lines(citizenship=country, gender=gender, work_genre=genre, work_language=lang, work_min_year=min_year, work_max_year=max_year),
        is_advanced=True,
    )


def _task_writer_citizenship_gender(
    *, country_key: str, gender_key: str, k: int
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    gender = _bk_ent(GENDERS, gender_key)
    qru = f"Назови {k} писателей с гражданством: {country['ru']} и полом: {gender['ru']}."
    qen = f"Name {k} writers whose citizenship is {country['en']} and gender is {gender['en']}."
    return BookTask(
        template_id="books_writer_citizenship_gender",
        template_family="writers_basic",
        answer_kind="writer",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_writer_base(citizenship=country, gender=gender),
        answer_lines=_bk_writer_answer_lines("item"),
        where_lines=_bk_writer_common_lines(citizenship=country, gender=gender),
        is_advanced=False,
    )


def _task_writer_award_citizenship_gender(
    *, country_key: str, gender_key: str, award_key: str, k: int
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    gender = _bk_ent(GENDERS, gender_key)
    award = _bk_ent(AWARDS, award_key)
    qru = f"Назови {k} писателей с гражданством: {country['ru']}, полом: {gender['ru']} и наградой: {award['ru']}."
    qen = f"Name {k} writers whose citizenship is {country['en']}, gender is {gender['en']}, and who received the {award['en']}."
    return BookTask(
        template_id="books_writer_citizenship_gender_award",
        template_family="writers_awards",
        answer_kind="writer",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_writer_base(citizenship=country, gender=gender, award=award),
        answer_lines=_bk_writer_answer_lines("item"),
        where_lines=_bk_writer_common_lines(citizenship=country, gender=gender, award=award),
        is_advanced=True,
    )


def _task_work_author_award_genre_language_year(
    *, award_key: str, genre_key: str, lang_key: str, min_year: Optional[int], max_year: Optional[int], k: int
) -> BookTask:
    award = _bk_ent(AWARDS, award_key)
    genre = _bk_ent(GENRES, genre_key)
    lang = _bk_ent(LANGUAGES, lang_key)
    yru = _bk_year_phrase_ru(min_year, max_year)
    yen = _bk_year_phrase_en(min_year, max_year)
    qru = f"Назови {k} литературных произведений жанра «{genre['ru']}» на языке: {lang['ru']}, опубликованных {yru}, у авторов с наградой: {award['ru']}."
    qen = f"Name {k} literary works in the {genre['en']} genre whose language is {lang['en']}, published {yen}, and whose authors received the {award['en']}."
    return BookTask(
        template_id="books_work_author_award_genre_language_year",
        template_family="works_author_awards",
        answer_kind="literary work",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_work_base(genre=genre, language=lang, author_award=award, min_year=min_year, max_year=max_year),
        answer_lines=_bk_literary_work_answer_lines("item"),
        where_lines=_bk_work_common_lines(genre=genre, language=lang, author_award=award, min_year=min_year, max_year=max_year),
        is_advanced=True,
    )


def _task_work_setting_adaptation_year(
    *, city_key: str, lang_key: Optional[str], pub_max_year: Optional[int], adapt_min_year: int, k: int
) -> BookTask:
    city = _bk_ent(CITIES, city_key)
    lang = _bk_ent(LANGUAGES, lang_key) if lang_key else None
    lang_ru = f" на языке: {lang['ru']}," if lang else ""
    lang_en = f" whose language is {lang['en']}," if lang else ""
    pub_ru = f" опубликованных не позднее {int(pub_max_year)} года," if pub_max_year is not None else ""
    pub_en = f" published no later than {int(pub_max_year)}," if pub_max_year is not None else ""
    qru = f"Назови {k} литературных произведений,{lang_ru}{pub_ru} где место действия связано с городом {city['ru']} и есть экранизация, выпущенная не ранее {int(adapt_min_year)} года."
    qen = f"Name {k} literary works{lang_en}{pub_en} whose narrative location is connected to {city['en']} and that have a screen adaptation released no earlier than {int(adapt_min_year)}."
    lines = []
    if lang is not None:
        lines.append(f"?item wdt:P407 wd:{lang['qid']} .")
    if pub_max_year is not None:
        lines.extend(_bk_date_lines("item", None, pub_max_year))
    lines.extend([
        f"?item wdt:P840/wdt:P131* wd:{city['qid']} .",
        "?adaptation wdt:P144 ?item .",
        f"VALUES ?adaptationType {{ wd:{Q_FILM} wd:{Q_TV_SERIES} }}",
        "?adaptation wdt:P31/wdt:P279* ?adaptationType .",
        "?adaptation wdt:P577 ?adaptationDate .",
        f"FILTER(YEAR(?adaptationDate) >= {int(adapt_min_year)}) .",
    ])
    extra = {"narrative_location": city["en"], "has_screen_adaptation": True, "screen_adaptation_year_min": int(adapt_min_year)}
    return BookTask(
        template_id="books_work_setting_adaptation_year",
        template_family="works_setting_adaptation",
        answer_kind="literary work",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_work_base(language=lang, max_year=pub_max_year, extra=extra),
        answer_lines=_bk_literary_work_answer_lines("item"),
        where_lines=lines,
        is_advanced=True,
    )


def _task_work_adapted_author_country_language_year(
    *, country_key: str, lang_key: str, min_year: Optional[int], max_year: Optional[int], adapt_min_year: int, k: int
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    lang = _bk_ent(LANGUAGES, lang_key)
    yru = _bk_year_phrase_ru(min_year, max_year)
    yen = _bk_year_phrase_en(min_year, max_year)
    qru = f"Назови {k} литературных произведений на языке: {lang['ru']}, опубликованных {yru}, у авторов с гражданством: {country['ru']}, у которых есть экранизация не ранее {int(adapt_min_year)} года."
    qen = f"Name {k} literary works whose language is {lang['en']}, published {yen}, whose authors have citizenship of {country['en']}, and that have a screen adaptation released no earlier than {int(adapt_min_year)}."
    lines = _bk_work_common_lines(language=lang, author_country=country, min_year=min_year, max_year=max_year)
    lines.extend([
        "?adaptation wdt:P144 ?item .",
        f"VALUES ?adaptationType {{ wd:{Q_FILM} wd:{Q_TV_SERIES} }}",
        "?adaptation wdt:P31/wdt:P279* ?adaptationType .",
        "?adaptation wdt:P577 ?adaptationDate .",
        f"FILTER(YEAR(?adaptationDate) >= {int(adapt_min_year)}) .",
    ])
    return BookTask(
        template_id="books_work_adapted_author_country_language_year",
        template_family="works_adaptations",
        answer_kind="literary work",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_work_base(language=lang, author_country=country, min_year=min_year, max_year=max_year, extra={"has_screen_adaptation": True, "screen_adaptation_year_min": int(adapt_min_year)}),
        answer_lines=_bk_literary_work_answer_lines("item"),
        where_lines=lines,
        is_advanced=True,
    )


def _task_writer_with_adapted_work(
    *, country_key: str, gender_key: str, genre_key: str, lang_key: str, adapt_min_year: int, k: int
) -> BookTask:
    country = _bk_ent(COUNTRIES, country_key)
    gender = _bk_ent(GENDERS, gender_key)
    genre = _bk_ent(GENRES, genre_key)
    lang = _bk_ent(LANGUAGES, lang_key)
    qru = f"Назови {k} писателей с гражданством: {country['ru']} и полом: {gender['ru']}, у которых есть произведение жанра «{genre['ru']}» на языке: {lang['ru']} с экранизацией не ранее {int(adapt_min_year)} года."
    qen = f"Name {k} writers whose citizenship is {country['en']} and gender is {gender['en']}, and who authored a {genre['en']} work in {lang['en']} with a screen adaptation released no earlier than {int(adapt_min_year)}."
    return BookTask(
        template_id="books_writer_with_adapted_work",
        template_family="writers_adaptations",
        answer_kind="writer",
        requested_count=k,
        query_text_ru=qru,
        query_text_en=qen,
        constraints=_bk_constraints_writer_base(citizenship=country, gender=gender, work_genre=genre, work_language=lang, adapted_work_after_year=adapt_min_year),
        answer_lines=_bk_writer_answer_lines("item"),
        where_lines=_bk_writer_common_lines(citizenship=country, gender=gender, work_genre=genre, work_language=lang, adapted_work_after_year=adapt_min_year),
        is_advanced=True,
    )


## 7. Candidate plans

Plans are deliberately mixed by answer type and pattern to avoid one-template dominance. The expanded candidate pool supports the L1-L5 run.


In [7]:
def _bk_task_from_spec(spec: Dict[str, Any]) -> BookTask:
    """Build a BookTask from a candidate spec.

    Candidate specs may include scheduling/debug-only keys such as
    ``priority`` or planned diversity hints.  Older versions passed those
    through to the template builder and crashed with
    ``TypeError: unexpected keyword argument 'priority'``.  Filter kwargs by
    the actual builder signature so auxiliary spec fields can never break
    generation again.
    """
    import inspect

    name = spec["template"]
    builder = BOOKS_TEMPLATE_BUILDERS[name]
    allowed = set(inspect.signature(builder).parameters)
    kwargs = {k: v for k, v in spec.items() if k in allowed}
    return builder(**kwargs)


BOOKS_TEMPLATE_BUILDERS: Dict[str, Callable[..., BookTask]] = {
    "work_genre_language_year": _task_work_genre_language_year,
    "work_author_country_genre_year": _task_work_author_country_genre_year,
    "work_author_country_language_year": _task_work_author_country_language_year,
    "work_author_gender_country_genre_language_year": _task_work_author_gender_country_genre_language_year,
    "writer_citizenship_gender": _task_writer_citizenship_gender,
    "writer_citizenship_gender_work_genre_language": _task_writer_citizenship_gender_work_genre_language,
    "writer_award_citizenship_gender": _task_writer_award_citizenship_gender,
    "work_author_award_genre_language_year": _task_work_author_award_genre_language_year,
    "work_setting_adaptation_year": _task_work_setting_adaptation_year,
    "work_adapted_author_country_language_year": _task_work_adapted_author_country_language_year,
    "writer_with_adapted_work": _task_writer_with_adapted_work,
}

# Candidate specs are intentionally redundant: the generator tries alternatives until one
# returns a complete <=100-answer gold set.
BOOKS_CANDIDATES_BY_COMPLEXITY: Dict[str, List[Dict[str, Any]]] = {
    "L1": [
        {"template": "work_genre_language_year", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1950, "max_year": 1969, "k": 5},
        {"template": "work_genre_language_year", "genre_key": "detective_fiction", "lang_key": "english", "min_year": 1900, "max_year": 1960, "k": 5},
        {"template": "work_author_country_genre_year", "country_key": "russia", "genre_key": "science_fiction", "min_year": 1900, "max_year": 1999, "k": 5},
        {"template": "work_author_country_language_year", "country_key": "france", "lang_key": "french", "min_year": 1850, "max_year": 1950, "k": 5},
        {"template": "writer_citizenship_gender_work_genre_language", "country_key": "united_states", "gender_key": "female", "genre_key": "science_fiction", "lang_key": "english", "k": 5},
        {"template": "work_genre_language_year", "genre_key": "children_literature", "lang_key": "english", "min_year": 1900, "max_year": 1970, "k": 5},
        {"template": "work_author_country_language_year", "country_key": "japan", "lang_key": "japanese", "min_year": 1900, "max_year": 2000, "k": 5},
        {"template": "work_author_country_genre_year", "country_key": "united_kingdom", "genre_key": "fantasy", "min_year": 1900, "max_year": 2000, "k": 5},
    ],
    "L2": [
        {"template": "work_author_country_genre_year", "country_key": "united_states", "genre_key": "science_fiction", "min_year": 1950, "max_year": 1989, "k": 4},
        {"template": "work_author_country_language_year", "country_key": "united_kingdom", "lang_key": "english", "min_year": 1850, "max_year": 1930, "k": 4},
        {"template": "writer_citizenship_gender_work_genre_language", "country_key": "united_kingdom", "gender_key": "female", "genre_key": "detective_fiction", "lang_key": "english", "k": 4},
        {"template": "work_setting_adaptation_year", "city_key": "london", "lang_key": "english", "pub_max_year": 1950, "adapt_min_year": 1990, "k": 4},
        {"template": "work_author_country_genre_year", "country_key": "france", "genre_key": "historical_novel", "min_year": 1800, "max_year": 1950, "k": 4},
        {"template": "work_author_country_language_year", "country_key": "russia", "lang_key": "russian", "min_year": 1850, "max_year": 1950, "k": 4},
        {"template": "work_genre_language_year", "genre_key": "horror_fiction", "lang_key": "english", "min_year": 1800, "max_year": 1970, "k": 4},
        {"template": "writer_award_citizenship_gender", "country_key": "united_states", "gender_key": "male", "award_key": "pulitzer_fiction", "k": 4},
        {"template": "work_setting_adaptation_year", "city_key": "paris", "lang_key": "french", "pub_max_year": 1950, "adapt_min_year": 1980, "k": 4},
        {"template": "work_author_country_genre_year", "country_key": "japan", "genre_key": "detective_fiction", "min_year": 1920, "max_year": 2010, "k": 4},
    ],
    "L3": [
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "united_states", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1960, "max_year": 2010, "k": 4},
        {"template": "work_author_award_genre_language_year", "award_key": "hugo_best_novel", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1950, "max_year": 2010, "k": 4},
        {"template": "work_setting_adaptation_year", "city_key": "new_york_city", "lang_key": "english", "pub_max_year": 1980, "adapt_min_year": 1990, "k": 4},
        {"template": "writer_with_adapted_work", "country_key": "united_states", "gender_key": "male", "genre_key": "science_fiction", "lang_key": "english", "adapt_min_year": 1980, "k": 4},
        {"template": "work_adapted_author_country_language_year", "country_key": "united_kingdom", "lang_key": "english", "min_year": 1850, "max_year": 1980, "adapt_min_year": 1990, "k": 4},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "united_kingdom", "genre_key": "detective_fiction", "lang_key": "english", "min_year": 1900, "max_year": 2000, "k": 4},
        {"template": "work_author_award_genre_language_year", "award_key": "booker_prize", "genre_key": "historical_novel", "lang_key": "english", "min_year": 1960, "max_year": 2020, "k": 4},
        {"template": "writer_citizenship_gender_work_genre_language", "country_key": "france", "gender_key": "male", "genre_key": "science_fiction", "lang_key": "french", "min_year": 1850, "max_year": 2000, "k": 4},
        {"template": "work_setting_adaptation_year", "city_key": "moscow", "lang_key": "russian", "pub_max_year": 1950, "adapt_min_year": 1960, "k": 4},
        {"template": "work_author_country_genre_year", "country_key": "ireland", "genre_key": "historical_novel", "min_year": 1900, "max_year": 2020, "k": 4},
    ],
    "L4": [
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "canada", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1970, "max_year": 2020, "k": 3},
        {"template": "work_adapted_author_country_language_year", "country_key": "united_states", "lang_key": "english", "min_year": 1900, "max_year": 1980, "adapt_min_year": 2000, "k": 3},
        {"template": "writer_with_adapted_work", "country_key": "united_kingdom", "gender_key": "female", "genre_key": "detective_fiction", "lang_key": "english", "adapt_min_year": 1980, "k": 3},
        {"template": "work_author_award_genre_language_year", "award_key": "nobel_literature", "genre_key": "historical_novel", "lang_key": "english", "min_year": 1900, "max_year": 2020, "k": 3},
        {"template": "work_setting_adaptation_year", "city_key": "london", "lang_key": "english", "pub_max_year": 1930, "adapt_min_year": 2000, "k": 3},
        {"template": "writer_citizenship_gender_work_genre_language", "country_key": "russia", "gender_key": "female", "genre_key": "science_fiction", "lang_key": "russian", "min_year": 1950, "max_year": 2020, "k": 3},
        {"template": "work_author_country_genre_year", "country_key": "germany", "genre_key": "horror_fiction", "min_year": 1800, "max_year": 2020, "k": 3},
        {"template": "work_setting_adaptation_year", "city_key": "paris", "lang_key": "french", "pub_max_year": 1930, "adapt_min_year": 1990, "k": 3},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "japan", "genre_key": "crime_fiction", "lang_key": "japanese", "min_year": 1950, "max_year": 2020, "k": 3},
        {"template": "work_adapted_author_country_language_year", "country_key": "france", "lang_key": "french", "min_year": 1850, "max_year": 1970, "adapt_min_year": 1980, "k": 3},
    ],
    "L5": [
        {"template": "work_author_award_genre_language_year", "award_key": "hugo_best_novel", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1960, "max_year": 1999, "k": 3},
        {"template": "work_setting_adaptation_year", "city_key": "london", "lang_key": "english", "pub_max_year": 1910, "adapt_min_year": 2010, "k": 3},
        {"template": "writer_with_adapted_work", "country_key": "united_states", "gender_key": "female", "genre_key": "science_fiction", "lang_key": "english", "adapt_min_year": 1990, "k": 3},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "united_states", "genre_key": "fantasy", "lang_key": "english", "min_year": 1980, "max_year": 2020, "k": 3},
        {"template": "work_adapted_author_country_language_year", "country_key": "japan", "lang_key": "japanese", "min_year": 1950, "max_year": 2020, "adapt_min_year": 2000, "k": 3},
        {"template": "work_author_award_genre_language_year", "award_key": "booker_prize", "genre_key": "historical_novel", "lang_key": "english", "min_year": 1980, "max_year": 2020, "k": 3},
        {"template": "writer_citizenship_gender_work_genre_language", "country_key": "canada", "gender_key": "female", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1970, "max_year": 2020, "k": 3},
        {"template": "work_setting_adaptation_year", "city_key": "new_york_city", "lang_key": "english", "pub_max_year": 1950, "adapt_min_year": 2000, "k": 3},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "france", "genre_key": "historical_novel", "lang_key": "french", "min_year": 1900, "max_year": 2020, "k": 3},
        {"template": "writer_with_adapted_work", "country_key": "united_kingdom", "gender_key": "male", "genre_key": "fantasy", "lang_key": "english", "adapt_min_year": 2000, "k": 3},
    ],
}

# Preferred template order by level. Diversity is decided before WDQS calls.
BOOKS_TEMPLATE_PLAN: Dict[str, List[str]] = {
    "L1": [
        "work_genre_language_year",
        "work_author_country_genre_year",
        "writer_citizenship_gender_work_genre_language",
        "work_author_country_language_year",
        "work_genre_language_year",
    ],
    "L2": [
        "work_author_country_genre_year",
        "work_setting_adaptation_year",
        "writer_citizenship_gender_work_genre_language",
        "work_author_country_language_year",
        "work_genre_language_year",
        "work_author_country_genre_year",
        "writer_award_citizenship_gender",
        "work_setting_adaptation_year",
    ],
    "L3": [
        "work_author_gender_country_genre_language_year",
        "work_author_award_genre_language_year",
        "work_setting_adaptation_year",
        "writer_with_adapted_work",
        "work_adapted_author_country_language_year",
        "writer_citizenship_gender_work_genre_language",
    ] * 2,
    "L4": [
        "work_adapted_author_country_language_year",
        "writer_with_adapted_work",
        "work_author_award_genre_language_year",
        "work_setting_adaptation_year",
        "work_author_gender_country_genre_language_year",
    ] * 3,
    "L5": [
        "work_author_award_genre_language_year",
        "work_setting_adaptation_year",
        "writer_with_adapted_work",
        "work_author_gender_country_genre_language_year",
        "work_adapted_author_country_language_year",
    ] * 3,
}


# ---------------------------------------------------------------------------
# Expanded L1-L5 candidate pool
# ---------------------------------------------------------------------------
# The hand-written seed list above is kept for stability. The code below expands it
# into a large, schema-safe candidate pool so that the L1-L5 run does not recycle
# the same few constraints. Diversity is still decided before WDQS calls.

BOOKS_COUNTRY_LANGUAGE_PAIRS: List[Tuple[str, str]] = [
    ("united_states", "english"),
    ("united_kingdom", "english"),
    ("canada", "english"),
    ("ireland", "english"),
    ("russia", "russian"),
    ("france", "french"),
    ("germany", "german"),
    ("japan", "japanese"),
    ("italy", "italian"),
    ("spain", "spanish"),
    ("poland", "polish"),
    ("argentina", "spanish"),
]

BOOKS_PRIMARY_COUNTRIES = ["united_states", "united_kingdom", "russia", "france", "germany", "japan", "canada", "ireland"]
BOOKS_PRIMARY_GENRES = [
    "science_fiction",
    "fantasy",
    "detective_fiction",
    "horror_fiction",
    "historical_novel",
    "dystopian_fiction",
    "children_literature",
    "crime_fiction",
    "adventure_fiction",
    "romance_novel",
]
BOOKS_PRIMARY_LANGUAGES = ["english", "russian", "french", "german", "japanese", "spanish", "italian", "polish"]
BOOKS_YEAR_WINDOWS = [
    (1800, 1849),
    (1850, 1899),
    (1900, 1939),
    (1940, 1969),
    (1970, 1989),
    (1990, 2010),
    (2011, 2024),
]
BOOKS_WIDE_YEAR_WINDOWS = [
    (1800, 1950),
    (1850, 2000),
    (1900, 2024),
    (1950, 2024),
]
BOOKS_ADAPTATION_MIN_YEARS = [1950, 1970, 1980, 1990, 2000, 2010]
BOOKS_CITY_LANGUAGE_PAIRS = [
    ("london", "english"),
    ("new_york_city", "english"),
    ("dublin", "english"),
    ("paris", "french"),
    ("moscow", "russian"),
    ("saint_petersburg", "russian"),
    ("tokyo", "japanese"),
    ("berlin", "german"),
]

BOOKS_LANGUAGE_ROTATION_BY_COMPLEXITY: Dict[str, List[str]] = {
    # Soft rotation only: prioritize candidates with the desired language,
    # but fall back to other languages if WDQS/quality gates reject them.
    # This prevents the accepted dataset from accidentally skewing to
    # Russian/French/English just because those candidates are easiest under
    # the strict RU+EN label policy.
    "L1": ["russian", "french", "german", "english", "japanese", "spanish", "italian", "polish", "russian", "german"],
    "L2": ["english", "russian", "french", "german", "japanese", "spanish", "italian", "polish"],
    "L3": ["english", "french", "russian", "german", "japanese", "spanish", "italian", "polish"],
    "L4": ["english", "german", "french", "russian", "japanese", "spanish", "italian", "polish"],
    "L5": ["english", "french", "german", "russian", "japanese", "spanish", "italian", "polish"],
}

def _bk_spec_language_key(spec: Dict[str, Any]) -> Optional[str]:
    """Return the explicit or inferred language key for specs that constrain language."""
    if spec.get("lang_key"):
        return str(spec.get("lang_key"))
    if spec.get("template") == "work_author_country_genre_year" and spec.get("country_key"):
        return _bk_default_language_for_country(str(spec.get("country_key")))
    return None

def _bk_preferred_language_for(complexity: str, idx: int) -> Optional[str]:
    rotation = BOOKS_LANGUAGE_ROTATION_BY_COMPLEXITY.get(complexity) or []
    if not rotation:
        return None
    return rotation[(int(idx) - 1) % len(rotation)]


BOOKS_GENRE_ROTATION_BY_COMPLEXITY: Dict[str, List[str]] = {
    # Soft genre rotation. It does not force a failing genre, but it makes the
    # generator try under-represented genres first instead of repeatedly accepting
    # the easiest science-fiction / fantasy / children's-literature candidates.
    "L1": [
        "fantasy", "science_fiction", "children_literature", "detective_fiction",
        "historical_novel", "adventure_fiction", "dystopian_fiction",
        "crime_fiction", "horror_fiction", "romance_novel",
    ],
    "L2": [
        "historical_novel", "crime_fiction", "adventure_fiction", "detective_fiction",
        "fantasy", "science_fiction", "children_literature", "horror_fiction",
        "dystopian_fiction", "romance_novel",
    ],
    "L3": [
        "detective_fiction", "historical_novel", "crime_fiction", "adventure_fiction",
        "science_fiction", "fantasy", "horror_fiction", "dystopian_fiction",
        "children_literature", "romance_novel",
    ],
    "L4": [
        "historical_novel", "detective_fiction", "crime_fiction", "horror_fiction",
        "fantasy", "adventure_fiction", "science_fiction", "dystopian_fiction",
        "children_literature", "romance_novel",
    ],
    "L5": [
        "crime_fiction", "historical_novel", "detective_fiction", "adventure_fiction",
        "horror_fiction", "fantasy", "science_fiction", "dystopian_fiction",
        "children_literature", "romance_novel",
    ],
}

# Soft cap is used only as a sorting penalty, not as a hard rejection. This
# preserves robustness under noisy Wikidata while avoiding early genre collapse.
BOOKS_GENRE_OVERUSE_SOFT_CAP_BY_LEVEL: Dict[str, int] = {
    "L1": 2,
    "L2": 3,
    "L3": 5,
    "L4": 5,
    "L5": 6,
}

def _bk_spec_genre_key(spec: Dict[str, Any]) -> Optional[str]:
    g = spec.get("genre_key")
    return str(g) if g else None

def _bk_preferred_genre_for(complexity: str, idx: int) -> Optional[str]:
    rotation = BOOKS_GENRE_ROTATION_BY_COMPLEXITY.get(complexity) or []
    if not rotation:
        return None
    return rotation[(int(idx) - 1) % len(rotation)]

def _bk_genre_key_from_public_value(value: Optional[str]) -> Optional[str]:
    if not value:
        return None
    v = str(value)
    for key, meta in GENRES.items():
        if meta.get("en") == v:
            return key
    return None

def _bk_record_genre_key(ex: BenchmarkExample) -> Optional[str]:
    meta = getattr(ex, "gold_collection_meta", None) or {}
    g = meta.get("candidate_genre_key") or meta.get("planned_genre_key")
    if g:
        return str(g)
    return _bk_genre_key_from_public_value((getattr(ex, "constraints", None) or {}).get("genre"))


BOOKS_GENDER_ROTATION_BY_COMPLEXITY: Dict[str, List[Optional[str]]] = {
    # Soft rotation for gendered author/writer templates. None means no gender preference.
    # This prevents a level from accepting a long run of only female- or only male-constrained queries
    # while still allowing non-gender templates to pass normally.
    "L3": ["female", "male", None, "male", "female", None],
    "L4": ["male", "female", None, "female", "male", None],
    "L5": ["female", "male", None, "female", "male", None],
}

def _bk_spec_gender_key(spec: Dict[str, Any]) -> Optional[str]:
    g = spec.get("gender_key")
    return str(g) if g else None

def _bk_preferred_gender_for(complexity: str, idx: int) -> Optional[str]:
    rotation = BOOKS_GENDER_ROTATION_BY_COMPLEXITY.get(complexity) or []
    if not rotation:
        return None
    return rotation[(int(idx) - 1) % len(rotation)]


def _bk_balanced_preferred_gender_for(
    complexity: str,
    idx: int,
    level_gender_counts: Optional[Counter] = None,
) -> Optional[str]:
    """Return a preferred author gender with male/female balancing for L3-L5.

    Earlier L3-only runs tended to accept whichever gendered profile happened to
    be easiest first. For a benchmark this is bad: the user sees a wall of
    `author_gender=female` or `author_gender=male` examples. This helper keeps
    gendered author constraints balanced while still allowing no-gender templates
    to be selected by the template rotation.
    """
    base = _bk_preferred_gender_for(complexity, idx)
    if not BOOKS_BALANCE_GENDERED_AUTHOR_QUERIES or complexity not in {"L3", "L4", "L5"}:
        return base
    counts = level_gender_counts or Counter()
    female = int(counts.get("female", 0))
    male = int(counts.get("male", 0))
    # If one side is already ahead, force the underrepresented gender.
    if female - male >= int(BOOKS_GENDER_BALANCE_MAX_DIFF):
        return "male"
    if male - female >= int(BOOKS_GENDER_BALANCE_MAX_DIFF):
        return "female"
    return base


def _bk_spec_gender_balance_score(spec: Dict[str, Any], level_gender_counts: Optional[Counter]) -> int:
    """Small score used to prefer the underrepresented gendered profile."""
    gender = _bk_spec_gender_key(spec)
    if gender not in {"female", "male"} or not level_gender_counts:
        return 0
    return int(level_gender_counts.get(gender, 0))



def _bk_spec_key(spec: Dict[str, Any]) -> str:
    return json.dumps(spec, ensure_ascii=False, sort_keys=True)


def _bk_dedupe_specs(specs: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    seen: set = set()
    for spec in specs:
        key = _bk_spec_key(spec)
        if key in seen:
            continue
        seen.add(key)
        out.append(dict(spec))
    return out


def _bk_add_spec(target: Dict[str, List[Dict[str, Any]]], level: str, **spec: Any) -> None:
    target.setdefault(level, []).append(dict(spec))


def _bk_make_expanded_candidates() -> Dict[str, List[Dict[str, Any]]]:
    expanded: Dict[str, List[Dict[str, Any]]] = {k: list(v) for k, v in BOOKS_CANDIDATES_BY_COMPLEXITY.items()}

    # L1: broad but still semantically constrained, mostly works and a few writer queries.
    for genre_key in ["science_fiction", "detective_fiction", "fantasy", "children_literature", "historical_novel"]:
        for lang_key in BOOKS_PRIMARY_LANGUAGES:
            for min_year, max_year in [(1850, 1950), (1900, 1999), (1950, 2024)]:
                _bk_add_spec(expanded, "L1", template="work_genre_language_year", genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=5)

    for country_key, lang_key in BOOKS_COUNTRY_LANGUAGE_PAIRS:
        for min_year, max_year in [(1850, 1950), (1900, 2000), (1950, 2024)]:
            _bk_add_spec(expanded, "L1", template="work_author_country_language_year", country_key=country_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=5)

    for country_key, lang_key in [("united_states", "english"), ("united_kingdom", "english"), ("france", "french"), ("russia", "russian"), ("japan", "japanese")]:
        for gender_key in ["female", "male"]:
            for genre_key in ["science_fiction", "detective_fiction", "fantasy", "historical_novel"]:
                _bk_add_spec(expanded, "L1", template="writer_citizenship_gender_work_genre_language", country_key=country_key, gender_key=gender_key, genre_key=genre_key, lang_key=lang_key, k=5)

    # L2: author nationality, genre/language, adaptations and award-winner writers.
    for country_key, lang_key in BOOKS_COUNTRY_LANGUAGE_PAIRS:
        for genre_key in BOOKS_PRIMARY_GENRES:
            for min_year, max_year in [(1850, 1950), (1900, 1989), (1950, 2024)]:
                _bk_add_spec(expanded, "L2", template="work_author_country_genre_year", country_key=country_key, genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=4)
                _bk_add_spec(expanded, "L2", template="work_author_country_language_year", country_key=country_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=4)

    for city_key, lang_key in BOOKS_CITY_LANGUAGE_PAIRS:
        for pub_max_year in [1900, 1930, 1950, 1980]:
            for adapt_min_year in [1970, 1990, 2000]:
                _bk_add_spec(expanded, "L2", template="work_setting_adaptation_year", city_key=city_key, lang_key=lang_key, pub_max_year=pub_max_year, adapt_min_year=adapt_min_year, k=4)

    for award_key in AWARDS.keys():
        for country_key in BOOKS_PRIMARY_COUNTRIES:
            for gender_key in ["female", "male"]:
                _bk_add_spec(expanded, "L2", template="writer_award_citizenship_gender", country_key=country_key, gender_key=gender_key, award_key=award_key, k=4)

    # L3: complex multi-hop work/writer patterns, mostly with four requested answers.
    for country_key, lang_key in BOOKS_COUNTRY_LANGUAGE_PAIRS:
        for genre_key in ["science_fiction", "fantasy", "detective_fiction", "historical_novel", "crime_fiction", "dystopian_fiction"]:
            for min_year, max_year in [(1900, 1970), (1950, 2000), (1970, 2024)]:
                for gender_key in ["female", "male"]:
                    _bk_add_spec(expanded, "L3", template="work_author_gender_country_genre_language_year", country_key=country_key, gender_key=gender_key, genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=4)
                _bk_add_spec(expanded, "L3", template="writer_citizenship_gender_work_genre_language", country_key=country_key, gender_key="female", genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=4)
                _bk_add_spec(expanded, "L3", template="writer_citizenship_gender_work_genre_language", country_key=country_key, gender_key="male", genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=4)

    for award_key in AWARDS.keys():
        for genre_key in ["science_fiction", "fantasy", "historical_novel", "detective_fiction", "crime_fiction"]:
            for lang_key in ["english", "french", "russian", "japanese"]:
                for min_year, max_year in [(1900, 1980), (1950, 2024), (1980, 2024)]:
                    _bk_add_spec(expanded, "L3", template="work_author_award_genre_language_year", award_key=award_key, genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=4)

    for country_key, lang_key in BOOKS_COUNTRY_LANGUAGE_PAIRS:
        for min_year, max_year in [(1850, 1950), (1900, 1980), (1950, 2024)]:
            for adapt_min_year in [1970, 1990, 2000]:
                _bk_add_spec(expanded, "L3", template="work_adapted_author_country_language_year", country_key=country_key, lang_key=lang_key, min_year=min_year, max_year=max_year, adapt_min_year=adapt_min_year, k=4)

    # L4/L5: stricter versions with adaptations, awards, female authors, and writer answers.
    for level, k in [("L4", 3), ("L5", 3)]:
        for country_key, lang_key in BOOKS_COUNTRY_LANGUAGE_PAIRS:
            for genre_key in ["science_fiction", "fantasy", "detective_fiction", "historical_novel", "crime_fiction", "horror_fiction", "adventure_fiction"]:
                for min_year, max_year in [(1900, 1960), (1950, 1990), (1970, 2024), (1990, 2024)]:
                    for gender_key in ["female", "male"]:
                        _bk_add_spec(expanded, level, template="work_author_gender_country_genre_language_year", country_key=country_key, gender_key=gender_key, genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=k)

        for award_key in AWARDS.keys():
            for genre_key in ["science_fiction", "fantasy", "historical_novel", "detective_fiction", "crime_fiction", "dystopian_fiction"]:
                for lang_key in ["english", "french", "russian", "japanese", "german", "spanish"]:
                    for min_year, max_year in [(1900, 1970), (1950, 2000), (1980, 2024)]:
                        _bk_add_spec(expanded, level, template="work_author_award_genre_language_year", award_key=award_key, genre_key=genre_key, lang_key=lang_key, min_year=min_year, max_year=max_year, k=k)

        for city_key, lang_key in BOOKS_CITY_LANGUAGE_PAIRS:
            for pub_max_year in [1900, 1930, 1950, 1980, None]:
                for adapt_min_year in [1970, 1990, 2000, 2010]:
                    _bk_add_spec(expanded, level, template="work_setting_adaptation_year", city_key=city_key, lang_key=lang_key, pub_max_year=pub_max_year, adapt_min_year=adapt_min_year, k=k)

        for country_key, lang_key in BOOKS_COUNTRY_LANGUAGE_PAIRS:
            for min_year, max_year in [(1850, 1950), (1900, 1980), (1950, 2024), (1980, 2024)]:
                for adapt_min_year in [1970, 1990, 2000, 2010]:
                    _bk_add_spec(expanded, level, template="work_adapted_author_country_language_year", country_key=country_key, lang_key=lang_key, min_year=min_year, max_year=max_year, adapt_min_year=adapt_min_year, k=k)

        for country_key, lang_key in BOOKS_COUNTRY_LANGUAGE_PAIRS:
            for gender_key in ["female", "male"]:
                for genre_key in ["science_fiction", "fantasy", "detective_fiction", "historical_novel", "crime_fiction"]:
                    for adapt_min_year in [1970, 1990, 2000, 2010]:
                        _bk_add_spec(expanded, level, template="writer_with_adapted_work", country_key=country_key, gender_key=gender_key, genre_key=genre_key, lang_key=lang_key, adapt_min_year=adapt_min_year, k=k)
                        _bk_add_spec(expanded, level, template="writer_citizenship_gender_work_genre_language", country_key=country_key, gender_key=gender_key, genre_key=genre_key, lang_key=lang_key, min_year=1950, max_year=2024, k=k)

    return {level: _bk_dedupe_specs(specs) for level, specs in expanded.items()}


BOOKS_CANDIDATES_BY_COMPLEXITY = _bk_make_expanded_candidates()


# ---------------------------------------------------------------------------
# Level calibration patch
# ---------------------------------------------------------------------------
# Books/L1 without a year bound (e.g. genre + language only) looks deceptively
# simple but often has a huge hidden Wikidata universe and incomplete-feeling
# public gold after RU+EN label filtering. For this domain, L1 means direct
# constraints on the answer item (no author/adaptation/award hop), but every
# L1 work query is now bounded by publication year to keep gold sets compact,
# reproducible and fast to collect.
BOOKS_L1_BASE_SPECS = [
    # First pass is deliberately genre-balanced. These are tried before the
    # large combinatorial pool and prevent the beginning of the file from
    # collapsing into science fiction / fantasy / children's literature only.
    {"template": "work_genre_language_year", "genre_key": "fantasy", "lang_key": "russian", "min_year": 1900, "max_year": 1999, "k": 5, "priority": 1},
    {"template": "work_genre_language_year", "genre_key": "science_fiction", "lang_key": "french", "min_year": 1850, "max_year": 1899, "k": 5, "priority": 2},
    {"template": "work_genre_language_year", "genre_key": "children_literature", "lang_key": "german", "min_year": 1900, "max_year": 1999, "k": 5, "priority": 3},
    {"template": "work_genre_language_year", "genre_key": "detective_fiction", "lang_key": "russian", "min_year": 1900, "max_year": 1999, "k": 5, "priority": 4},
    {"template": "work_genre_language_year", "genre_key": "historical_novel", "lang_key": "french", "min_year": 1800, "max_year": 1899, "k": 5, "priority": 5},
    {"template": "work_genre_language_year", "genre_key": "adventure_fiction", "lang_key": "french", "min_year": 1800, "max_year": 1950, "k": 5, "priority": 6},
    {"template": "work_genre_language_year", "genre_key": "dystopian_fiction", "lang_key": "english", "min_year": 1900, "max_year": 2024, "k": 5, "priority": 7},
    {"template": "work_genre_language_year", "genre_key": "crime_fiction", "lang_key": "english", "min_year": 1900, "max_year": 1989, "k": 5, "priority": 8},
    {"template": "work_genre_language_year", "genre_key": "horror_fiction", "lang_key": "english", "min_year": 1800, "max_year": 1970, "k": 5, "priority": 9},
    {"template": "work_genre_language_year", "genre_key": "romance_novel", "lang_key": "english", "min_year": 1800, "max_year": 2024, "k": 5, "priority": 10},
    {"template": "work_genre_language_year", "genre_key": "science_fiction", "lang_key": "polish", "min_year": 1900, "max_year": 1989, "k": 5, "priority": 11},
    {"template": "work_genre_language_year", "genre_key": "historical_novel", "lang_key": "spanish", "min_year": 1900, "max_year": 2024, "k": 5, "priority": 12},
]
_l1_specs = list(BOOKS_L1_BASE_SPECS)
_l1_genres = [
    "science_fiction", "detective_fiction", "fantasy", "historical_novel",
    "children_literature", "crime_fiction", "adventure_fiction", "dystopian_fiction",
    "horror_fiction", "romance_novel"
]
_l1_windows = [
    (1800, 1849), (1850, 1899), (1900, 1939), (1940, 1969),
    (1970, 1989), (1990, 2024), (1900, 1999), (1950, 2024)
]
_priority = 100
for _genre_key in _l1_genres:
    for _lang_key in BOOKS_PRIMARY_LANGUAGES:
        for _min_year, _max_year in _l1_windows:
            _priority += 1
            _l1_specs.append({
                "template": "work_genre_language_year",
                "genre_key": _genre_key,
                "lang_key": _lang_key,
                "min_year": _min_year,
                "max_year": _max_year,
                "k": 5,
                "priority": _priority,
            })
BOOKS_CANDIDATES_BY_COMPLEXITY["L1"] = _bk_dedupe_specs(_l1_specs)
del _l1_specs, _l1_genres, _l1_windows, _priority, _genre_key, _lang_key, _min_year, _max_year
BOOKS_L1_ALLOWED_TEMPLATES = {"work_genre_language_year"}
BOOKS_L1_ALLOWED_TASK_IDS = {"books_work_genre_language_year"}


# L1-L5 plan. Repeated template names are intentional;
# candidate-level shuffling and used public constraint signatures keep actual examples diverse.
BOOKS_TEMPLATE_PLAN = {
    "L1": [
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
        "work_genre_language_year",
    ],
    "L2": [
        "work_author_country_genre_year",
        "work_author_country_language_year",
        "work_setting_adaptation_year",
        "writer_citizenship_gender_work_genre_language",
        "writer_award_citizenship_gender",
    ] * 4,
    "L3": [
        "work_author_gender_country_genre_language_year",
        "work_author_award_genre_language_year",
        "work_setting_adaptation_year",
        "writer_with_adapted_work",
        "work_adapted_author_country_language_year",
        "writer_citizenship_gender_work_genre_language",
        "work_author_country_genre_year",
    ] * 5,
    "L4": [
        "work_adapted_author_country_language_year",
        "writer_with_adapted_work",
        "work_author_award_genre_language_year",
        "work_setting_adaptation_year",
        "work_author_gender_country_genre_language_year",
        "writer_citizenship_gender_work_genre_language",
        "work_author_country_language_year",
        "writer_with_adapted_work",
    ] * 5,
    "L5": [
        "work_author_award_genre_language_year",
        "work_setting_adaptation_year",
        "writer_with_adapted_work",
        "work_author_gender_country_genre_language_year",
        "work_adapted_author_country_language_year",
        "writer_citizenship_gender_work_genre_language",
        "work_author_award_genre_language_year",
        "writer_with_adapted_work",
        "work_setting_adaptation_year",
    ] * 5,
}


def _bk_template_constraints_signature(task: BookTask) -> Tuple[Any, ...]:
    # Keep exact duplicate public tasks out of the dataset without doing post-WDQS expensive checks.
    c = task.constraints or {}
    return (task.template_id, json.dumps(c, ensure_ascii=False, sort_keys=True))


def _bk_candidate_specs_for(
    complexity: str,
    preferred_template: Optional[str],
    rng: random.Random,
    preferred_lang_key: Optional[str] = None,
    preferred_gender_key: Optional[str] = None,
    preferred_genre_key: Optional[str] = None,
    level_genre_counts: Optional[Counter] = None,
    level_gender_counts: Optional[Counter] = None,
) -> List[Dict[str, Any]]:
    specs = list(BOOKS_CANDIDATES_BY_COMPLEXITY.get(complexity, []))
    rng.shuffle(specs)

    def score(spec: Dict[str, Any]) -> Tuple[int, int, int, int, int, int, int]:
        template_score = 0 if (preferred_template and spec.get("template") == preferred_template) else 1
        genre = _bk_spec_genre_key(spec)

        overuse_score = 0
        if genre and level_genre_counts:
            soft_cap = int(BOOKS_GENRE_OVERUSE_SOFT_CAP_BY_LEVEL.get(complexity, 999))
            overuse_score = max(0, int(level_genre_counts.get(genre, 0)) - soft_cap + 1)

        lang_score = 0
        if preferred_lang_key:
            lang = _bk_spec_language_key(spec)
            if lang == preferred_lang_key:
                lang_score = 0
            elif lang is None:
                lang_score = 1
            else:
                lang_score = 2

        gender_score = 0
        if preferred_gender_key:
            gender = _bk_spec_gender_key(spec)
            if gender == preferred_gender_key:
                gender_score = 0
            elif gender is None:
                gender_score = 1
            else:
                gender_score = 2

        genre_score = 0
        if preferred_genre_key:
            if genre == preferred_genre_key:
                genre_score = 0
            elif genre is None:
                genre_score = 1
            else:
                genre_score = 2

        priority_score = int(spec.get("priority", 10_000))
        gender_balance_score = _bk_spec_gender_balance_score(spec, level_gender_counts)
        # Priority and overuse are intentionally before genre preference in the
        # fallback ordering. Gender balance is also early: if female L3 records
        # are already ahead, male candidates get tried first, and vice versa.
        return (template_score, overuse_score, gender_balance_score, priority_score, lang_score, gender_score, genre_score)

    specs.sort(key=score)
    if not preferred_genre_key:
        return specs[:BOOKS_PER_TEMPLATE_CANDIDATE_LIMIT]

    preferred = [s for s in specs if _bk_spec_genre_key(s) == preferred_genre_key]
    fallback = [s for s in specs if _bk_spec_genre_key(s) != preferred_genre_key]
    max_pref = max(0, int(BOOKS_MAX_PREFERRED_GENRE_TRIES_PER_SLOT))
    ordered = preferred[:max_pref] + fallback
    return ordered[:BOOKS_PER_TEMPLATE_CANDIDATE_LIMIT]



In [8]:

# ---------------------------------------------------------------------------
# v34 validated fast seeds for L3
# ---------------------------------------------------------------------------
# These specs are copied from records that already passed the strict WDQS gates
# during earlier runs. They are tried first to avoid the "0 accepted for many
# minutes" problem. Gold is NOT hardcoded: each spec is rebuilt and queried via
# WDQS like any other candidate.
BOOKS_VALIDATED_FAST_SEED_SPECS_BY_LEVEL: Dict[str, List[Dict[str, Any]]] = {
    "L3": [
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "united_kingdom", "genre_key": "detective_fiction", "lang_key": "english", "min_year": 1950, "max_year": 2000, "k": 4, "priority": -100},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "france", "genre_key": "detective_fiction", "lang_key": "french", "min_year": 1900, "max_year": 1970, "k": 4, "priority": -99},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "russia", "genre_key": "fantasy", "lang_key": "russian", "min_year": 1950, "max_year": 2000, "k": 4, "priority": -98},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "germany", "genre_key": "fantasy", "lang_key": "german", "min_year": 1970, "max_year": 2024, "k": 4, "priority": -97},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "united_states", "genre_key": "detective_fiction", "lang_key": "english", "min_year": 1970, "max_year": 2024, "k": 4, "priority": -96},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "italy", "genre_key": "fantasy", "lang_key": "italian", "min_year": 1970, "max_year": 2024, "k": 4, "priority": -95},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "female", "country_key": "united_kingdom", "genre_key": "fantasy", "lang_key": "english", "min_year": 1900, "max_year": 1970, "k": 4, "priority": -94},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "canada", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1900, "max_year": 1970, "k": 4, "priority": -93},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "russia", "genre_key": "science_fiction", "lang_key": "russian", "min_year": 1950, "max_year": 2000, "k": 4, "priority": -92},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "ireland", "genre_key": "science_fiction", "lang_key": "english", "min_year": 1950, "max_year": 2000, "k": 4, "priority": -91},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "germany", "genre_key": "fantasy", "lang_key": "german", "min_year": 1950, "max_year": 2000, "k": 4, "priority": -90},
        {"template": "work_author_gender_country_genre_language_year", "gender_key": "male", "country_key": "argentina", "genre_key": "fantasy", "lang_key": "spanish", "min_year": 1900, "max_year": 1970, "k": 4, "priority": -89},
    ],
    "L4": [],
    "L5": [],
}

def _bk_try_seed_spec_as_example(
    spec: Dict[str, Any],
    complexity: str,
    public_idx: int,
    used_signatures: set,
) -> Optional[BenchmarkExample]:
    """Try one known-good seed spec with the full normal WDQS/quality path."""
    try:
        task = _bk_task_from_spec(spec)
    except Exception:
        return None
    sig = _bk_template_constraints_signature(task)
    if sig in used_signatures:
        return None
    ex = _bk_make_example(task, complexity, public_idx, 1)
    if ex is None:
        return None
    if len(ex.gold_answer_qids) < ex.requested_count or len(ex.gold_answer_qids) > BOOKS_MAX_GOLD_ALLOWED:
        return None
    if not _bk_public_constraints_ok(ex.constraints):
        return None
    meta = dict(ex.gold_collection_meta or {})
    if meta.get("strict_sparql_filter_guard") is not True:
        return None
    if meta.get("wdqs_exact_count_after_label_filter") != len(ex.gold_answer_qids):
        return None
    used_signatures.add(sig)
    meta.update({
        "generator_version": BOOKS_GENERATOR_VERSION,
        "runner_seed_path": "validated_fast_seed_spec",
        "candidate_template": spec.get("template"),
        "candidate_language_key": _bk_spec_language_key(spec),
        "candidate_genre_key": _bk_spec_genre_key(spec),
        "candidate_gender_key": _bk_spec_gender_key(spec),
    })
    ex.gold_collection_meta = meta
    return ex


## 8. Main generator


In [9]:
def generate_books_example(
    complexity: str,
    idx: int,
    rng: Optional[random.Random] = None,
    max_attempts: int = BOOKS_GENERATOR_MAX_ATTEMPTS,
    used_signatures: Optional[set] = None,
    rejected_signatures: Optional[set] = None,
    level_genre_counts: Optional[Counter] = None,
    level_gender_counts: Optional[Counter] = None,
) -> BenchmarkExample:
    """Generate one accepted books example without stalling on a bad slot.

    v26 could fail after a few accepted L3 records because the runner reused the
    same diversity slot, while the global rejected-signature cache made most
    subsequent candidates count as ``known_bad_constraints``. This version:

    * treats known-bad/used signatures as cheap skips, not WDQS attempts;
    * tries several language/gender/genre diversity profiles for the same public id;
    * keeps rejected signatures local by default for L1-L5;
    * only counts real WDQS probes toward ``max_attempts``.
    """
    rng = rng or BOOKS_RNG
    used_signatures = used_signatures if used_signatures is not None else set()
    rejected_signatures = rejected_signatures if rejected_signatures is not None else set()

    plan = list(BOOKS_TEMPLATE_PLAN.get(complexity, []))
    if not plan:
        plan = [None]
    # Add a final all-template fallback pass.
    preferred_sequence = plan + [None]

    skipped_reasons = Counter()
    local_seen_signatures: set = set()
    local_rejected_signatures: set = set()
    wdqs_probe_attempts = 0
    scanned_specs = 0
    started_at = time.monotonic()

    profile_offsets = list(range(max(1, int(BOOKS_DIVERSITY_PROFILE_OFFSETS))))
    # Deterministic but not identical for every stuck public id.
    if len(profile_offsets) > 2:
        first = profile_offsets[:2]
        rest = profile_offsets[2:]
        rng.shuffle(rest)
        profile_offsets = first + rest

    for offset in profile_offsets:
        preferred_language = _bk_preferred_language_for(complexity, idx + offset)
        preferred_gender = _bk_balanced_preferred_gender_for(complexity, idx + offset, level_gender_counts)
        preferred_genre = _bk_preferred_genre_for(complexity, idx + offset)

        for preferred in preferred_sequence:
            if (time.monotonic() - started_at) > float(BOOKS_MAX_SECONDS_PER_ACCEPTED_RECORD):
                skipped_reasons["record_wall_time_budget_exceeded"] += 1
                raise RuntimeError(
                    f"Failed to generate books example {complexity}/{idx}: "
                    f"time budget exceeded after {wdqs_probe_attempts} WDQS probes; "
                    f"scanned={scanned_specs}; skipped={dict(skipped_reasons)}"
                )

            specs = _bk_candidate_specs_for(
                complexity,
                preferred,
                rng,
                preferred_lang_key=preferred_language,
                preferred_gender_key=preferred_gender,
                preferred_genre_key=preferred_genre,
                level_genre_counts=level_genre_counts,
                level_gender_counts=level_gender_counts,
            )

            for spec in specs:
                scanned_specs += 1
                try:
                    task = _bk_task_from_spec(spec)
                except Exception as e:
                    skipped_reasons[f"task_build_error:{type(e).__name__}"] += 1
                    continue

                sig = _bk_template_constraints_signature(task)
                if sig in local_seen_signatures:
                    skipped_reasons["local_duplicate_candidate"] += 1
                    continue
                local_seen_signatures.add(sig)

                if sig in used_signatures:
                    skipped_reasons["duplicate_public_constraints"] += 1
                    continue
                if sig in local_rejected_signatures:
                    skipped_reasons["known_bad_constraints_local"] += 1
                    continue
                if BOOKS_USE_GLOBAL_REJECTED_SIGNATURE_CACHE and sig in rejected_signatures:
                    skipped_reasons["known_bad_constraints_global"] += 1
                    continue

                if wdqs_probe_attempts >= int(max_attempts):
                    raise RuntimeError(
                        f"Failed to generate books example {complexity}/{idx}: "
                        f"max WDQS probes reached ({wdqs_probe_attempts}/{max_attempts}); "
                        f"scanned={scanned_specs}; skipped={dict(skipped_reasons)}"
                    )
                if (time.monotonic() - started_at) > float(BOOKS_MAX_SECONDS_PER_ACCEPTED_RECORD):
                    skipped_reasons["record_wall_time_budget_exceeded"] += 1
                    raise RuntimeError(
                        f"Failed to generate books example {complexity}/{idx}: "
                        f"time budget exceeded after {wdqs_probe_attempts} WDQS probes; "
                        f"scanned={scanned_specs}; skipped={dict(skipped_reasons)}"
                    )

                wdqs_probe_attempts += 1
                ex = _bk_make_example(task, complexity, idx, wdqs_probe_attempts)
                if ex is None:
                    local_rejected_signatures.add(sig)
                    if BOOKS_USE_GLOBAL_REJECTED_SIGNATURE_CACHE:
                        rejected_signatures.add(sig)
                    skipped_reasons["wdqs_or_quality_gate_failed"] += 1
                    continue

                # Extra invariants before returning.
                if len(ex.gold_answer_qids) < ex.requested_count:
                    local_rejected_signatures.add(sig)
                    skipped_reasons["not_enough_gold_after_build"] += 1
                    continue
                if len(ex.gold_answer_qids) > BOOKS_MAX_GOLD_ALLOWED:
                    local_rejected_signatures.add(sig)
                    skipped_reasons["too_many_gold_after_build"] += 1
                    continue
                if not _bk_public_constraints_ok(ex.constraints):
                    local_rejected_signatures.add(sig)
                    skipped_reasons["dirty_constraints"] += 1
                    continue
                if "wdqs_exact_count_after_label_filter" not in (ex.gold_collection_meta or {}):
                    local_rejected_signatures.add(sig)
                    skipped_reasons["missing_exact_count_meta"] += 1
                    continue

                used_signatures.add(sig)
                meta = dict(ex.gold_collection_meta or {})
                meta.update({
                    "planned_template_id": preferred,
                    "planned_language_key": preferred_language,
                    "planned_genre_key": preferred_genre,
                    "candidate_template": spec.get("template"),
                    "candidate_language_key": _bk_spec_language_key(spec),
                    "candidate_genre_key": _bk_spec_genre_key(spec),
                    "planned_gender_key": preferred_gender,
                    "candidate_gender_key": _bk_spec_gender_key(spec),
                    "skipped_reasons_before_success": dict(skipped_reasons),
                    "scanned_candidate_specs_before_success": scanned_specs,
                    "diversity_profile_offset_used": offset,
                    "global_rejected_cache_enabled": BOOKS_USE_GLOBAL_REJECTED_SIGNATURE_CACHE,
                    "gender_balance_counts_before_success": dict(level_gender_counts or {}),
                })
                ex.gold_collection_meta = meta
                return ex

    raise RuntimeError(
        f"Failed to generate books example {complexity}/{idx} after {wdqs_probe_attempts} WDQS probes; "
        f"scanned={scanned_specs}; skipped={dict(skipped_reasons)}"
    )


# Compatibility with the project-level domain registry, if present.
try:
    DOMAIN_GENERATORS
except NameError:
    DOMAIN_GENERATORS = {}
DOMAIN_GENERATORS["books"] = generate_books_example
DOMAIN_GENERATORS["04_books"] = generate_books_example


## 9. Sanity checks

These checks run without WDQS calls and catch the common regressions from previous domains.


In [10]:
def _bk_run_sanity_checks() -> None:
    assert BOOKS_GENERATOR_VERSION == "v36_130_l1_l5_resumable"
    assert BOOKS_TARGET_PLAN == {"L1": 10, "L2": 20, "L3": 30, "L4": 35, "L5": 35}, BOOKS_TARGET_PLAN
    assert BOOKS_TARGET_TOTAL == 130
    assert set(BOOKS_TARGET_PLAN) == {"L1", "L2", "L3", "L4", "L5"}
    assert BOOKS_TARGET_PLAN["L3"] + BOOKS_TARGET_PLAN["L4"] + BOOKS_TARGET_PLAN["L5"] == 100
    assert BOOKS_TARGET_PLAN["L1"] + BOOKS_TARGET_PLAN["L2"] == 30

    # Label policy: no OPTIONAL/fallback labels.
    q = _bk_build_select_query(_bk_literary_work_answer_lines("item"), ["?item wdt:P407 wd:Q1860 ."], limit=BOOKS_PROBE_LIMIT)
    assert "OPTIONAL" not in q
    assert "BOUND(?itemLabelRu)" not in q
    assert '?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .' in q
    assert '?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .' in q
    assert "LIMIT 101" in q
    cq = _bk_build_count_query(_bk_literary_work_answer_lines("item"), ["?item wdt:P407 wd:Q1860 ."], item_var="item")
    assert "COUNT(DISTINCT ?item)" in cq and "LIMIT" not in cq

    # Strict work filters and earliest-publication guard.
    genre_task = _task_work_genre_language_year(genre_key="science_fiction", lang_key="russian", min_year=1970, max_year=2024, k=5)
    genre_where = "\n".join(genre_task.where_lines)
    assert "wdt:P136/wdt:P279*" in genre_where
    assert "FILTER NOT EXISTS" in genre_where and "?earlierPublicationDate" in genre_where
    answer_s = "\n".join(genre_task.answer_lines)
    for qid in (Q_EDITION_OR_TRANSLATION, Q_BOOK_SERIES, Q_SERIES_OF_CREATIVE_WORKS, Q_WEBSITE, Q_WIKI):
        assert f"wd:{qid}" in answer_s, qid

    # Historical novel must be strict novel, not broad literary work.
    hist_task = _task_work_genre_language_year(genre_key="historical_novel", lang_key="french", min_year=1800, max_year=1899, k=5)
    hist_where = "\n".join(hist_task.where_lines)
    assert hist_task.constraints.get("kind") == "novel", hist_task.constraints
    assert f"wd:{Q_NOVEL}" in hist_where and f"wd:{Q_SHORT_STORY}" in hist_where and f"wd:{Q_NOVELLA}" in hist_where

    # Constraints must stay public/clean.
    task = _task_work_genre_language_year(genre_key="science_fiction", lang_key="english", min_year=1950, max_year=1969, k=5)
    assert _bk_public_constraints_ok(task.constraints), task.constraints
    bad_tokens = ["qid", "template_id", "label_en", "requires_en_label", "max_gold_allowed"]
    serialized = json.dumps(task.constraints, ensure_ascii=False).lower()
    assert not any(tok in serialized for tok in bad_tokens), serialized

    # Fail-fast/direct WDQS settings for L1-L5 generation.
    assert 0.30 <= BOOKS_MIN_LABEL_RETENTION_RATIO < 1
    assert BOOKS_GENERATOR_MAX_ATTEMPTS >= 100
    assert BOOKS_PER_TEMPLATE_CANDIDATE_LIMIT >= 600
    assert BOOKS_WDQS_FAST_TIMEOUT_SECONDS <= 10
    assert BOOKS_WDQS_FAST_MAX_RETRIES == 1
    assert BOOKS_MAX_SECONDS_PER_ACCEPTED_RECORD <= 300
    assert "requests" in _bk_rows_from_wdqs.__code__.co_names and "get" in _bk_rows_from_wdqs.__code__.co_names
    assert BOOKS_MAX_PREFERRED_GENRE_TRIES_PER_SLOT <= 1
    assert BOOKS_REQUIRE_UNIQUE_PUBLIC_GOLD_LABELS is True
    assert BOOKS_USE_GLOBAL_REJECTED_SIGNATURE_CACHE is False
    assert BOOKS_MAX_FAILED_GENERATE_CALLS_PER_LEVEL >= 10
    assert BOOKS_OUTPUT_PATH.name == "books.jsonl"
    assert BOOKS_AUDIT_PATH.name == "books.audit.json"

    # L1/L2 are enabled; L3-L5 remain the emphasized majority.
    for lvl in ("L1", "L2", "L3", "L4", "L5"):
        assert lvl in BOOKS_TEMPLATE_PLAN, BOOKS_TEMPLATE_PLAN
        assert BOOKS_CANDIDATES_BY_COMPLEXITY.get(lvl), f"empty candidate pool for {lvl}"
    l1_task = _task_work_genre_language_year(genre_key="science_fiction", lang_key="english", min_year=1950, max_year=1969, k=5)
    l2_task = _task_work_author_country_genre_year(country_key="united_states", genre_key="science_fiction", min_year=1950, max_year=1989, k=4)
    assert l1_task.requested_count == 5 and l2_task.requested_count == 4
    flat_plan = [x for xs in BOOKS_TEMPLATE_PLAN.values() for x in xs]
    assert "writer_with_adapted_work" in flat_plan
    assert "work_setting_adaptation_year" in flat_plan
    assert "work_author_award_genre_language_year" in flat_plan
    assert "work_author_gender_country_genre_language_year" in flat_plan
    assert "work_female_author_country_genre_language_year" not in BOOKS_TEMPLATE_PLAN.get("L3", [])

    # Gender diversity for L1-L5 must not be female-only.
    generic_gender_specs = [s for s in BOOKS_CANDIDATES_BY_COMPLEXITY["L3"] if s.get("template") == "work_author_gender_country_genre_language_year"]
    assert generic_gender_specs and all(s.get("gender_key") in {"female", "male"} for s in generic_gender_specs)
    gendered_l3_specs = [s for s in BOOKS_CANDIDATES_BY_COMPLEXITY["L3"] if s.get("template") in {"work_author_gender_country_genre_language_year", "writer_citizenship_gender_work_genre_language", "writer_with_adapted_work"}]
    l3_genders = Counter(s.get("gender_key") for s in gendered_l3_specs if s.get("gender_key"))
    assert l3_genders.get("female", 0) > 0 and l3_genders.get("male", 0) > 0, l3_genders
    assert _bk_preferred_gender_for("L3", 1) == "female" and _bk_preferred_gender_for("L3", 2) == "male"
    assert _bk_balanced_preferred_gender_for("L3", 3, Counter({"female": 2, "male": 0})) == "male"
    assert _bk_balanced_preferred_gender_for("L3", 3, Counter({"female": 0, "male": 2})) == "female"
    assert BOOKS_BALANCE_GENDERED_AUTHOR_QUERIES is True
    assert BOOKS_USE_VALIDATED_FAST_SEEDS is True
    assert BOOKS_VALIDATED_FAST_SEED_SPECS_BY_LEVEL.get("L3")

_bk_run_sanity_checks()
print("✅ books v36 130 L1-L5 sanity checks passed")


✅ books v36 130 L1-L5 sanity checks passed


## 10. Dataset generation runner

Creates `out_wikidata_benchmark/domain_outputs/books.jsonl` and an audit file. Completed records are written incrementally.


In [11]:

def _bk_append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _bk_count_jsonl_rows(path: Path) -> int:
    path = Path(path)
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def _bk_write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


def _bk_dataclass_field_names() -> set:
    from dataclasses import fields
    return {f.name for f in fields(BenchmarkExample)}


def _bk_example_from_record_dict(record: Dict[str, Any]) -> BenchmarkExample:
    """Load an already written JSONL row back into BenchmarkExample.

    The loader intentionally keeps only fields declared on BenchmarkExample so
    old audit/debug-only keys cannot break resume. Missing optional fields keep
    their dataclass defaults.
    """
    allowed = _bk_dataclass_field_names()
    kwargs = {k: v for k, v in dict(record).items() if k in allowed}
    return BenchmarkExample(**kwargs)


def _bk_load_examples_jsonl(path: Path) -> List[BenchmarkExample]:
    path = Path(path)
    if not path.exists():
        return []
    out: List[BenchmarkExample] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                out.append(_bk_example_from_record_dict(json.loads(line)))
            except Exception as e:
                raise RuntimeError(f"Cannot resume: bad JSONL row {line_no} in {path}: {e}") from e
    return out


def _bk_load_jsonl_dicts(path: Path) -> List[Dict[str, Any]]:
    path = Path(path)
    if not path.exists():
        return []
    out: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    out.append(json.loads(line))
                except Exception:
                    pass
    return out


def _bk_record_signature(ex: BenchmarkExample) -> Tuple[Any, ...]:
    return (ex.template_id, json.dumps(ex.constraints or {}, ensure_ascii=False, sort_keys=True))


def _bk_public_index_from_id(example_id: str) -> Optional[int]:
    m = re.search(r"_(\d{4,})$", str(example_id or ""))
    if not m:
        return None
    try:
        return int(m.group(1))
    except Exception:
        return None


def _bk_next_public_index(records: Sequence[BenchmarkExample]) -> int:
    ids = [_bk_public_index_from_id(getattr(ex, "id", "")) for ex in records]
    ids = [x for x in ids if x is not None]
    if ids:
        return max(ids) + 1
    return len(records) + 1


def _bk_audit(records: Sequence[BenchmarkExample], skipped: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    by_level = Counter(ex.complexity for ex in records)
    by_kind = Counter((ex.constraints or {}).get("kind") for ex in records)
    by_template = Counter(ex.template_id for ex in records)
    by_genre = Counter(_bk_record_genre_key(ex) or "no_explicit_genre" for ex in records)
    by_level_genre = Counter((ex.complexity, _bk_record_genre_key(ex) or "no_explicit_genre") for ex in records)
    by_author_gender = Counter(((ex.constraints or {}).get("author_gender") or "no_author_gender") for ex in records)
    by_level_author_gender = Counter((ex.complexity, ((ex.constraints or {}).get("author_gender") or "no_author_gender")) for ex in records)
    gold_sizes = [len(ex.gold_answer_qids) for ex in records]
    target_by_level = dict(BOOKS_TARGET_PLAN)
    by_level_dict = dict(by_level)
    return {
        "domain": "books",
        "generator_version": BOOKS_GENERATOR_VERSION,
        "target_plan": target_by_level,
        "target_total": sum(target_by_level.values()),
        "record_count": len(records),
        "remaining_to_target_by_level": {lvl: max(0, int(target_by_level.get(lvl, 0)) - int(by_level_dict.get(lvl, 0))) for lvl in target_by_level},
        "skipped_count": len(skipped),
        "by_level": by_level_dict,
        "by_kind": dict(by_kind),
        "by_template": dict(by_template),
        "by_genre": dict(by_genre),
        "by_level_genre": {f"{lvl}:{genre}": n for (lvl, genre), n in by_level_genre.items()},
        "by_author_gender": dict(by_author_gender),
        "by_level_author_gender": {f"{lvl}:{gender}": n for (lvl, gender), n in by_level_author_gender.items()},
        "gold_size_min": min(gold_sizes) if gold_sizes else None,
        "gold_size_max": max(gold_sizes) if gold_sizes else None,
        "gold_size_avg": (sum(gold_sizes) / len(gold_sizes)) if gold_sizes else None,
        "quality_flags": {
            "schema_is_benchmark_example_dataclass": True,
            "clean_public_constraints": all(_bk_public_constraints_ok(ex.constraints or {}) for ex in records),
            "gold_size_lte_100": all(len(ex.gold_answer_qids) <= BOOKS_MAX_GOLD_ALLOWED for ex in records),
            "gold_size_gte_requested": all(len(ex.gold_answer_qids) >= ex.requested_count for ex in records),
            "no_en_cyrillic": all(not any(_CYRILLIC_RE_BOOKS.search(x or "") for x in ex.gold_answer_labels_en) for ex in records),
            "same_gold_array_lengths": all(len(ex.gold_answer_qids) == len(ex.gold_answer_labels_ru) == len(ex.gold_answer_labels_en) for ex in records),
            "has_exact_count_meta": all("wdqs_exact_count_after_label_filter" in (ex.gold_collection_meta or {}) for ex in records),
            "gold_returned_matches_exact_count": all((ex.gold_collection_meta or {}).get("wdqs_exact_count_after_label_filter") == len(ex.gold_answer_qids) for ex in records),
            "uses_earliest_publication_year_filter": True,
            "excludes_non_standalone_literary_work_noise": True,
            "all_records_have_generator_version": all((ex.gold_collection_meta or {}).get("generator_version") == BOOKS_GENERATOR_VERSION for ex in records),
            "all_records_pass_strict_filter_guard": all((ex.gold_collection_meta or {}).get("strict_sparql_filter_guard") is True for ex in records),
            "only_l1_l2_l3_l4_l5": all(ex.complexity in {"L1", "L2", "L3", "L4", "L5"} for ex in records),
        },
    }


def generate_books_dataset(
    target_plan: Optional[Dict[str, int]] = None,
    seed: int = BOOKS_SEED,
    output_path: Path = BOOKS_OUTPUT_PATH,
    overwrite: bool = False,
) -> List[BenchmarkExample]:
    """Generate the books domain dataset incrementally and resume safely.

    `overwrite=False` is the recommended mode. If `books.jsonl` already exists,
    the runner loads it, rebuilds diversity/duplicate state from the completed
    rows, and continues until the per-level target plan is reached. Every
    accepted record is appended immediately, so interrupted runs keep progress.
    """
    target_plan = dict(target_plan or BOOKS_TARGET_PLAN)
    if any(lvl not in {"L1", "L2", "L3", "L4", "L5"} for lvl in target_plan):
        raise ValueError(f"Books v36 target plan must use only L1-L5 keys, got: {target_plan}")

    rng = random.Random(seed)
    output_path = Path(output_path)
    skipped_path = BOOKS_SKIPPED_PATH
    audit_path = BOOKS_AUDIT_PATH
    checkpoint_path = BOOKS_CHECKPOINT_PATH

    if overwrite:
        for p in (output_path, skipped_path, audit_path, checkpoint_path):
            try:
                p.unlink()
            except FileNotFoundError:
                pass

    records: List[BenchmarkExample] = [] if overwrite else _bk_load_examples_jsonl(output_path)
    skipped: List[Dict[str, Any]] = [] if overwrite else _bk_load_jsonl_dicts(skipped_path)
    used_signatures: set = {_bk_record_signature(ex) for ex in records}
    rejected_signatures: set = set()
    level_genre_counts: Dict[str, Counter] = defaultdict(Counter)
    level_gender_counts: Dict[str, Counter] = defaultdict(Counter)
    incomplete_levels: Dict[str, Any] = {}

    for ex in records:
        level_genre_counts[ex.complexity][_bk_record_genre_key(ex) or "no_explicit_genre"] += 1
        level_gender_counts[ex.complexity][(ex.constraints or {}).get("author_gender") or "no_author_gender"] += 1

    existing_by_level = Counter(ex.complexity for ex in records)
    remaining_total = sum(max(0, int(target_plan.get(lvl, 0)) - int(existing_by_level.get(lvl, 0))) for lvl in target_plan)

    def write_checkpoint(current_level: Optional[str] = None, level_accepted: Optional[int] = None, level_target: Optional[int] = None) -> None:
        _bk_write_json(checkpoint_path, {
            "records_written": len(records),
            "jsonl_rows": _bk_count_jsonl_rows(output_path),
            "target_plan": target_plan,
            "current_level": current_level,
            "level_accepted": level_accepted,
            "level_target": level_target,
            "incomplete_levels": incomplete_levels,
            "updated_at": utc_now_z(),
        })

    def accept_record(ex: BenchmarkExample, complexity: str, public_idx: int, extra_meta: Optional[Dict[str, Any]] = None) -> int:
        ex.id = f"books_{complexity.lower()}_{public_idx:04d}"
        meta = dict(ex.gold_collection_meta or {})
        meta.update({
            "generator_version": BOOKS_GENERATOR_VERSION,
            "runner_version": BOOKS_GENERATOR_VERSION,
            "public_id_index": public_idx,
            "output_schema": "BenchmarkExample/asdict",
            "resumable_runner": True,
        })
        if extra_meta:
            meta.update(extra_meta)
        ex.gold_collection_meta = meta
        records.append(ex)
        _bk_append_jsonl(output_path, asdict(ex))
        used_signatures.add(_bk_record_signature(ex))
        level_genre_counts[complexity][_bk_record_genre_key(ex) or "no_explicit_genre"] += 1
        accepted_gender = (ex.constraints or {}).get("author_gender") or "no_author_gender"
        level_gender_counts[complexity][accepted_gender] += 1
        file_rows = _bk_count_jsonl_rows(output_path)
        if file_rows != len(records):
            raise RuntimeError(f"Progress/file mismatch: memory_records={len(records)}, jsonl_rows={file_rows}")
        return file_rows

    progress = tqdm(total=remaining_total, desc="books accepted") if (tqdm is not None and remaining_total > 0) else None
    global_idx = _bk_next_public_index(records)

    try:
        for complexity, target_n in target_plan.items():
            accepted_in_level = int(existing_by_level.get(complexity, 0))
            failed_calls_in_level = 0

            if accepted_in_level >= int(target_n):
                write_checkpoint(complexity, accepted_in_level, int(target_n))
                continue

            # 1) Fast seed pass. Seeds are still validated through WDQS and all
            # normal quality gates, so they never hardcode gold answers.
            if BOOKS_USE_VALIDATED_FAST_SEEDS:
                for seed_i, spec in enumerate(BOOKS_VALIDATED_FAST_SEED_SPECS_BY_LEVEL.get(complexity, []), 1):
                    if accepted_in_level >= int(target_n):
                        break
                    ex = _bk_try_seed_spec_as_example(spec, complexity, global_idx, used_signatures)
                    if ex is None:
                        skipped_record = {"complexity": complexity, "seed_index": seed_i, "reason": "seed_spec_failed_quality_or_wdqs", "created_at": utc_now_z()}
                        skipped.append(skipped_record)
                        _bk_append_jsonl(skipped_path, skipped_record)
                        continue
                    file_rows = accept_record(ex, complexity, global_idx, {"runner_seed_order": seed_i})
                    accepted_in_level += 1
                    global_idx += 1
                    write_checkpoint(complexity, accepted_in_level, int(target_n))
                    if progress is not None:
                        progress.update(1)
                        progress.set_postfix({
                            "level": complexity,
                            "level_ok": f"{accepted_in_level}/{target_n}",
                            "accepted": len(records),
                            "jsonl_rows": file_rows,
                            "skipped": len(skipped),
                            "genre": _bk_record_genre_key(ex) or "-",
                            "gender": (ex.constraints or {}).get("author_gender") or "-",
                            "gold": len(ex.gold_answer_qids),
                            "path": "seed",
                        })

            # 2) Main generator fallback. Bad candidate zones are skipped, but
            # the level is not abandoned until a generous failure budget is hit.
            while accepted_in_level < int(target_n):
                if failed_calls_in_level >= int(BOOKS_MAX_FAILED_GENERATE_CALLS_PER_LEVEL):
                    incomplete_levels[complexity] = {
                        "accepted": accepted_in_level,
                        "target": int(target_n),
                        "failed_calls": failed_calls_in_level,
                        "stopped_at": utc_now_z(),
                    }
                    skipped_record = {
                        "complexity": complexity,
                        "level_incomplete": True,
                        "level_accepted": accepted_in_level,
                        "level_target": int(target_n),
                        "reason": "max_failed_generate_calls_reached_no_crash",
                        "created_at": utc_now_z(),
                    }
                    skipped.append(skipped_record)
                    _bk_append_jsonl(skipped_path, skipped_record)
                    break

                profile_idx = global_idx + failed_calls_in_level * int(BOOKS_FAILED_SLOT_PROFILE_STRIDE)
                try:
                    ex = generate_books_example(
                        complexity,
                        profile_idx,
                        rng=rng,
                        used_signatures=used_signatures,
                        rejected_signatures=rejected_signatures,
                        level_genre_counts=level_genre_counts[complexity],
                        level_gender_counts=level_gender_counts[complexity],
                    )
                except Exception as e:
                    failed_calls_in_level += 1
                    rejected_signatures.clear()
                    skipped_record = {
                        "complexity": complexity,
                        "intended_public_id_index": global_idx,
                        "internal_profile_idx": profile_idx,
                        "level_accepted": accepted_in_level,
                        "level_target": int(target_n),
                        "failed_calls_in_level": failed_calls_in_level,
                        "max_failed_calls": int(BOOKS_MAX_FAILED_GENERATE_CALLS_PER_LEVEL),
                        "error": str(e),
                        "created_at": utc_now_z(),
                    }
                    skipped.append(skipped_record)
                    _bk_append_jsonl(skipped_path, skipped_record)
                    write_checkpoint(complexity, accepted_in_level, int(target_n))
                    if progress is not None:
                        progress.set_postfix({
                            "level": complexity,
                            "level_ok": f"{accepted_in_level}/{target_n}",
                            "accepted": len(records),
                            "jsonl_rows": _bk_count_jsonl_rows(output_path),
                            "skipped": len(skipped),
                            "last": "failed_slot",
                            "failed": failed_calls_in_level,
                        })
                    continue

                file_rows = accept_record(ex, complexity, global_idx, {
                    "internal_profile_idx": profile_idx,
                    "fallback_failed_calls_before_success": failed_calls_in_level,
                })
                accepted_in_level += 1
                global_idx += 1
                failed_calls_in_level = 0
                write_checkpoint(complexity, accepted_in_level, int(target_n))
                if progress is not None:
                    progress.update(1)
                    progress.set_postfix({
                        "level": complexity,
                        "level_ok": f"{accepted_in_level}/{target_n}",
                        "accepted": len(records),
                        "jsonl_rows": file_rows,
                        "skipped": len(skipped),
                        "genre": _bk_record_genre_key(ex) or "-",
                        "gender": (ex.constraints or {}).get("author_gender") or "-",
                        "gold": len(ex.gold_answer_qids),
                        "path": "fallback",
                    })

    finally:
        if progress is not None:
            progress.close()

    audit = _bk_audit(records, skipped)
    audit["incomplete_levels"] = incomplete_levels
    _bk_write_json(audit_path, audit)
    print(f"Wrote {len(records)} accepted books examples -> {output_path}")
    print(f"Audit: {audit_path}")
    if skipped:
        print(f"Skipped/failed candidates: {len(skipped)} -> {skipped_path}")
    if incomplete_levels:
        print(f"Incomplete levels: {incomplete_levels}")
    return records


if RUN_BOOKS_GENERATION:
    books_examples = generate_books_dataset(overwrite=False)
else:
    print("RUN_BOOKS_GENERATION=False; generation skipped.")


books accepted:  54%|█████▍    | 70/130 [7:47:34<6:40:46, 400.77s/it, level=L5, level_ok=0/35, accepted=70, jsonl_rows=70, skipped=156, last=failed_slot, failed=20]                                 

Wrote 70 accepted books examples -> out_wikidata_benchmark/domain_outputs/books.jsonl
Audit: out_wikidata_benchmark/domain_outputs/books.audit.json
Skipped/failed candidates: 157 -> out_wikidata_benchmark/domain_outputs/books.skipped.jsonl
Incomplete levels: {'L1': {'accepted': 8, 'target': 10, 'failed_calls': 20, 'stopped_at': '2026-05-22T18:22:56.340397Z'}, 'L2': {'accepted': 12, 'target': 20, 'failed_calls': 20, 'stopped_at': '2026-05-22T18:51:39.226148Z'}, 'L3': {'accepted': 15, 'target': 30, 'failed_calls': 20, 'stopped_at': '2026-05-22T19:16:11.214664Z'}, 'L5': {'accepted': 0, 'target': 35, 'failed_calls': 20, 'stopped_at': '2026-05-23T01:39:15.924611Z'}}
